# 🔬 Window Size Comparison: W3 vs W5 Embeddings

This section compares embeddings generated with **Window=3** vs **Window=5**.

**Goal**: Analyze how window size affects:
- Embedding quality
- Topic similarity scores
- Semantic coherence
- Data distribution

**Memory-Efficient Approach**: Process one topic at a time, clear memory, then create summary plots.

---

## 📦 Step 1: Import Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
import json
from scipy import stats
from scipy.spatial.distance import cosine
import warnings
warnings.filterwarnings('ignore')

# Memory management
def clear_memory():
    """Force garbage collection"""
    gc.collect()
    print(f"   ✅ Memory cleared")

# Dynamic Paths - Automatically detect workspace directory
CURRENT_DIR = Path.cwd()
print(f"Current working directory: {CURRENT_DIR}")

# Try to find the base directory intelligently
if 'Pre_Processin' in str(CURRENT_DIR):
    # We're in Pre_Processin folder
    BASE_DIR = CURRENT_DIR.parent / "Processed_Data"
    OUTPUT_DIR = CURRENT_DIR.parent / "Window_Comparison_Results"
else:
    # We're in main project folder
    BASE_DIR = CURRENT_DIR / "Processed_Data"
    OUTPUT_DIR = CURRENT_DIR / "Window_Comparison_Results"

W3_DIR = BASE_DIR / "Topic_Wise_w3"
W5_DIR = BASE_DIR / "Topic_Wise_w5"
TOPIC_WISE_OUTPUT_DIR = BASE_DIR / "Topic_Wise_w5"  # Primary output folder

# Create output directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOPIC_WISE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Topics to compare
TOPICS = ['War', 'Health', 'Technology', 'Climate', 'Economics']

print("✅ Libraries imported")
print(f"   Base Directory: {BASE_DIR}")
print(f"   W3 Directory: {W3_DIR}")
print(f"   W5 Directory: {W5_DIR}")
print(f"   Output Directory: {OUTPUT_DIR}")
print(f"   Topic-wise Output: {TOPIC_WISE_OUTPUT_DIR}")
print(f"   Topics: {', '.join(TOPICS)}")


## 🔧 Step 2: Helper Functions

In [ ]:
def parse_embedding(emb_str):
    """Parse embedding string to numpy array"""
    try:
        if isinstance(emb_str, str):
            emb_str = emb_str.strip('[]')
            values = [float(x.strip()) for x in emb_str.split(',')]
            return np.array(values, dtype=np.float32)
        elif isinstance(emb_str, (list, np.ndarray)):
            return np.array(emb_str, dtype=np.float32)
        else:
            return None
    except:
        return None

def compute_embedding_similarity(emb1, emb2):
    """Compute cosine similarity between two embeddings"""
    return 1 - cosine(emb1, emb2)

def compute_embedding_stats(embeddings_array):
    """Compute statistics for embedding array"""
    return {
        'mean_norm': float(np.linalg.norm(embeddings_array, axis=1).mean()),
        'std_norm': float(np.linalg.norm(embeddings_array, axis=1).std()),
        'mean_values': embeddings_array.mean(axis=0).mean(),
        'std_values': embeddings_array.std(axis=0).mean(),
        'sparsity': float((embeddings_array == 0).sum() / embeddings_array.size)
    }

print("✅ Helper functions defined")
print("   - parse_embedding()")
print("   - compute_embedding_similarity()")
print("   - compute_embedding_stats()")

## 🔄 Step 3: Topic-by-Topic Comparison (Memory Efficient)

**Process**: Load W3 → Load W5 → Compare → Save → Clear Memory → Next Topic

In [ ]:
# Storage for summary results (lightweight)
all_comparison_results = []

print("="*100)
print("STARTING TOPIC-BY-TOPIC COMPARISON (W3 vs W5)")
print("="*100)

for topic_idx, topic in enumerate(TOPICS, 1):
    print(f"\n{'#'*100}")
    print(f"# TOPIC {topic_idx}/{len(TOPICS)}: {topic}")
    print(f"{'#'*100}\n")
    
    # File paths
    w3_file = W3_DIR / f"{topic}.csv"
    w5_file = W5_DIR / f"{topic}.csv"
    
    if not w3_file.exists() or not w5_file.exists():
        print(f"⚠️  Files not found for {topic}")
        continue
    
    try:
        # ================================================================
        # LOAD W3 DATA
        # ================================================================
        print(f"📂 Loading W3 data...")
        df_w3 = pd.read_csv(w3_file)
        print(f"   W3: {len(df_w3):,} rows, {len(df_w3.columns)} columns")
        
        # ================================================================
        # LOAD W5 DATA
        # ================================================================
        print(f"📂 Loading W5 data...")
        df_w5 = pd.read_csv(w5_file)
        print(f"   W5: {len(df_w5):,} rows, {len(df_w5.columns)} columns")
        
        # ================================================================
        # BASIC COMPARISON
        # ================================================================
        print(f"\n{'─'*100}")
        print(f"BASIC STATISTICS COMPARISON")
        print(f"{'─'*100}")
        
        comparison = {
            'topic': topic,
            'w3_rows': len(df_w3),
            'w5_rows': len(df_w5),
            'w3_columns': len(df_w3.columns),
            'w5_columns': len(df_w5.columns)
        }
        
        # Check if topic similarity column exists
        if topic in df_w3.columns and topic in df_w5.columns:
            comparison['w3_topic_mean'] = float(df_w3[topic].mean())
            comparison['w3_topic_std'] = float(df_w3[topic].std())
            comparison['w5_topic_mean'] = float(df_w5[topic].mean())
            comparison['w5_topic_std'] = float(df_w5[topic].std())
            
            print(f"   {topic} Similarity Score (W3): {comparison['w3_topic_mean']:.4f} ± {comparison['w3_topic_std']:.4f}")
            print(f"   {topic} Similarity Score (W5): {comparison['w5_topic_mean']:.4f} ± {comparison['w5_topic_std']:.4f}")
            print(f"   Difference: {comparison['w5_topic_mean'] - comparison['w3_topic_mean']:.4f}")
        
        # ================================================================
        # EMBEDDING COMPARISON (Sample-based for memory)
        # ================================================================
        if 'w3_embedding' in df_w3.columns and 'w5_embedding' in df_w5.columns:
            print(f"\n{'─'*100}")
            print(f"EMBEDDING COMPARISON (Sample: 1000 rows)")
            print(f"{'─'*100}")
            
            # Sample 1000 rows for memory efficiency
            sample_size = min(1000, len(df_w3), len(df_w5))
            sample_indices = np.random.choice(min(len(df_w3), len(df_w5)), sample_size, replace=False)
            
            w3_embeddings = []
            w5_embeddings = []
            
            for idx in sample_indices:
                if idx < len(df_w3) and idx < len(df_w5):
                    emb_w3 = parse_embedding(df_w3.iloc[idx]['w3_embedding'])
                    emb_w5 = parse_embedding(df_w5.iloc[idx]['w5_embedding'])
                    
                    if emb_w3 is not None and emb_w5 is not None:
                        w3_embeddings.append(emb_w3)
                        w5_embeddings.append(emb_w5)
            
            if len(w3_embeddings) > 0:
                w3_array = np.array(w3_embeddings)
                w5_array = np.array(w5_embeddings)
                
                # Compute statistics
                w3_stats = compute_embedding_stats(w3_array)
                w5_stats = compute_embedding_stats(w5_array)
                
                print(f"   W3 Embedding Stats (n={len(w3_embeddings)}):")
                print(f"      Mean Norm: {w3_stats['mean_norm']:.4f}")
                print(f"      Std Norm:  {w3_stats['std_norm']:.4f}")
                print(f"      Sparsity:  {w3_stats['sparsity']:.4f}")
                
                print(f"   W5 Embedding Stats (n={len(w5_embeddings)}):")
                print(f"      Mean Norm: {w5_stats['mean_norm']:.4f}")
                print(f"      Std Norm:  {w5_stats['std_norm']:.4f}")
                print(f"      Sparsity:  {w5_stats['sparsity']:.4f}")
                
                # Compute pairwise similarity
                similarities = []
                for emb_w3, emb_w5 in zip(w3_embeddings[:100], w5_embeddings[:100]):
                    sim = compute_embedding_similarity(emb_w3, emb_w5)
                    similarities.append(sim)
                
                avg_similarity = np.mean(similarities)
                print(f"\n   Average W3-W5 Similarity: {avg_similarity:.4f}")
                
                comparison['w3_emb_norm_mean'] = w3_stats['mean_norm']
                comparison['w5_emb_norm_mean'] = w5_stats['mean_norm']
                comparison['w3_w5_similarity'] = float(avg_similarity)
                
                del w3_embeddings, w5_embeddings, w3_array, w5_array
        
        # ================================================================
        # CREATE TOPIC-SPECIFIC VISUALIZATION
        # ================================================================
        print(f"\n{'─'*100}")
        print(f"GENERATING VISUALIZATION")
        print(f"{'─'*100}")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'{topic} - W3 vs W5 Comparison', fontsize=16, fontweight='bold')
        
        # Plot 1: Topic Similarity Distribution
        if topic in df_w3.columns and topic in df_w5.columns:
            axes[0, 0].hist(df_w3[topic], bins=50, alpha=0.6, label='W3', color='blue', density=True)
            axes[0, 0].hist(df_w5[topic], bins=50, alpha=0.6, label='W5', color='red', density=True)
            axes[0, 0].set_xlabel(f'{topic} Similarity Score')
            axes[0, 0].set_ylabel('Density')
            axes[0, 0].set_title(f'{topic} Similarity Distribution')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
        
        # Plot 2: Box plot comparison
        if topic in df_w3.columns and topic in df_w5.columns:
            data_to_plot = [df_w3[topic].dropna(), df_w5[topic].dropna()]
            axes[0, 1].boxplot(data_to_plot, labels=['W3', 'W5'])
            axes[0, 1].set_ylabel(f'{topic} Similarity Score')
            axes[0, 1].set_title(f'{topic} Score Comparison')
            axes[0, 1].grid(True, alpha=0.3)
        
        # Plot 3: Row count comparison
        axes[1, 0].bar(['W3', 'W5'], [len(df_w3), len(df_w5)], color=['blue', 'red'], alpha=0.7)
        axes[1, 0].set_ylabel('Number of Rows')
        axes[1, 0].set_title('Data Volume Comparison')
        axes[1, 0].grid(True, alpha=0.3, axis='y')
        
        # Plot 4: Statistics summary text
        axes[1, 1].axis('off')
        summary_text = f"""
        Topic: {topic}
        
        W3 Statistics:
        • Rows: {len(df_w3):,}
        • Columns: {len(df_w3.columns)}
        """
        
        if topic in df_w3.columns:
            summary_text += f"• {topic} Mean: {df_w3[topic].mean():.4f}\n"
            summary_text += f"• {topic} Std: {df_w3[topic].std():.4f}\n"
        
        summary_text += f"""
        W5 Statistics:
        • Rows: {len(df_w5):,}
        • Columns: {len(df_w5.columns)}
        """
        
        if topic in df_w5.columns:
            summary_text += f"• {topic} Mean: {df_w5[topic].mean():.4f}\n"
            summary_text += f"• {topic} Std: {df_w5[topic].std():.4f}\n"
        
        axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
                       family='monospace')
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f'{topic}_w3_vs_w5_comparison.png', dpi=300, bbox_inches='tight')
        print(f"   ✅ Saved: {topic}_w3_vs_w5_comparison.png")
        plt.close(fig)
        
        # Save comparison results
        all_comparison_results.append(comparison)
        
        # ================================================================
        # CLEANUP MEMORY
        # ================================================================
        print(f"\n{'─'*100}")
        print(f"CLEANING UP MEMORY")
        print(f"{'─'*100}")
        
        del df_w3, df_w5, comparison
        clear_memory()
        
        print(f"\n✅ {topic} COMPLETE!\n")
        
    except Exception as e:
        print(f"\n❌ ERROR processing {topic}: {e}")
        import traceback
        traceback.print_exc()
        clear_memory()
        continue

print("\n" + "="*100)
print("TOPIC-BY-TOPIC COMPARISON COMPLETE")
print("="*100)
print(f"Processed {len(all_comparison_results)}/{len(TOPICS)} topics")

## 📊 Step 4: Summary Plots (All Topics Combined)

In [ ]:
print("="*100)
print("GENERATING SUMMARY PLOTS")
print("="*100)

# Convert results to DataFrame
summary_df = pd.DataFrame(all_comparison_results)

# Save summary table
summary_df.to_csv(OUTPUT_DIR / 'w3_vs_w5_summary.csv', index=False)
print(f"\n✅ Saved summary table: w3_vs_w5_summary.csv")

# Display summary
print("\n" + "─"*100)
print("SUMMARY TABLE")
print("─"*100)
print(summary_df.to_string(index=False))

# ============================================================================
# SUMMARY VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('W3 vs W5 Embeddings - Overall Comparison', fontsize=16, fontweight='bold')

# Plot 1: Row counts comparison
ax = axes[0, 0]
x = np.arange(len(summary_df))
width = 0.35
ax.bar(x - width/2, summary_df['w3_rows'], width, label='W3', alpha=0.8, color='blue')
ax.bar(x + width/2, summary_df['w5_rows'], width, label='W5', alpha=0.8, color='red')
ax.set_xlabel('Topics')
ax.set_ylabel('Number of Rows')
ax.set_title('Data Volume Comparison')
ax.set_xticks(x)
ax.set_xticklabels(summary_df['topic'], rotation=45)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Topic similarity mean comparison
if 'w3_topic_mean' in summary_df.columns and 'w5_topic_mean' in summary_df.columns:
    ax = axes[0, 1]
    x = np.arange(len(summary_df))
    ax.bar(x - width/2, summary_df['w3_topic_mean'], width, label='W3', alpha=0.8, color='blue')
    ax.bar(x + width/2, summary_df['w5_topic_mean'], width, label='W5', alpha=0.8, color='red')
    ax.set_xlabel('Topics')
    ax.set_ylabel('Mean Topic Similarity')
    ax.set_title('Topic Similarity Score Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df['topic'], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Topic similarity std comparison
if 'w3_topic_std' in summary_df.columns and 'w5_topic_std' in summary_df.columns:
    ax = axes[0, 2]
    ax.bar(x - width/2, summary_df['w3_topic_std'], width, label='W3', alpha=0.8, color='blue')
    ax.bar(x + width/2, summary_df['w5_topic_std'], width, label='W5', alpha=0.8, color='red')
    ax.set_xlabel('Topics')
    ax.set_ylabel('Std Deviation')
    ax.set_title('Topic Similarity Std Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df['topic'], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Embedding norm comparison
if 'w3_emb_norm_mean' in summary_df.columns and 'w5_emb_norm_mean' in summary_df.columns:
    ax = axes[1, 0]
    ax.bar(x - width/2, summary_df['w3_emb_norm_mean'], width, label='W3', alpha=0.8, color='blue')
    ax.bar(x + width/2, summary_df['w5_emb_norm_mean'], width, label='W5', alpha=0.8, color='red')
    ax.set_xlabel('Topics')
    ax.set_ylabel('Mean Embedding Norm')
    ax.set_title('Embedding Magnitude Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df['topic'], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

# Plot 5: W3-W5 Similarity
if 'w3_w5_similarity' in summary_df.columns:
    ax = axes[1, 1]
    colors = ['green' if s > 0.9 else 'orange' if s > 0.8 else 'red' 
              for s in summary_df['w3_w5_similarity']]
    ax.bar(summary_df['topic'], summary_df['w3_w5_similarity'], 
           alpha=0.8, color=colors)
    ax.set_xlabel('Topics')
    ax.set_ylabel('Cosine Similarity')
    ax.set_title('W3-W5 Embedding Similarity')
    ax.set_xticklabels(summary_df['topic'], rotation=45)
    ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='High (>0.9)')
    ax.axhline(y=0.8, color='orange', linestyle='--', alpha=0.5, label='Medium (>0.8)')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

# Plot 6: Summary statistics table
ax = axes[1, 2]
ax.axis('off')

summary_text = "Key Findings:\n\n"

if 'w3_topic_mean' in summary_df.columns and 'w5_topic_mean' in summary_df.columns:
    avg_diff = (summary_df['w5_topic_mean'] - summary_df['w3_topic_mean']).mean()
    summary_text += f"Avg Topic Score Diff (W5-W3):\n{avg_diff:+.4f}\n\n"

if 'w3_w5_similarity' in summary_df.columns:
    avg_sim = summary_df['w3_w5_similarity'].mean()
    summary_text += f"Avg W3-W5 Similarity:\n{avg_sim:.4f}\n\n"

summary_text += f"Topics Analyzed: {len(summary_df)}\n"
summary_text += f"\nW3 Total Rows: {summary_df['w3_rows'].sum():,}\n"
summary_text += f"W5 Total Rows: {summary_df['w5_rows'].sum():,}\n"

ax.text(0.1, 0.5, summary_text, fontsize=12, verticalalignment='center',
       family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'SUMMARY_w3_vs_w5_all_topics.png', dpi=300, bbox_inches='tight')
print(f"\n✅ Saved summary plot: SUMMARY_w3_vs_w5_all_topics.png")
plt.show()

print("\n" + "="*100)
print("✅ ALL COMPARISONS COMPLETE!")
print("="*100)
print(f"\n📁 All results saved to: {OUTPUT_DIR}")
print(f"\nGenerated files:")
print(f"   - w3_vs_w5_summary.csv (summary table)")
print(f"   - SUMMARY_w3_vs_w5_all_topics.png (combined plot)")
for topic in summary_df['topic']:
    print(f"   - {topic}_w3_vs_w5_comparison.png (individual plot)")
print("\n" + "="*100)

## 📈 Step 5: Statistical Analysis (Optional)

In [ ]:
# Statistical significance testing
print("="*100)
print("STATISTICAL ANALYSIS")
print("="*100)

if 'w3_topic_mean' in summary_df.columns and 'w5_topic_mean' in summary_df.columns:
    print("\n📊 Topic Similarity Score Analysis:")
    print("─"*100)
    
    for _, row in summary_df.iterrows():
        topic = row['topic']
        w3_mean = row['w3_topic_mean']
        w5_mean = row['w5_topic_mean']
        diff = w5_mean - w3_mean
        percent_change = (diff / w3_mean) * 100 if w3_mean > 0 else 0
        
        print(f"\n{topic}:")
        print(f"   W3 Mean: {w3_mean:.4f}")
        print(f"   W5 Mean: {w5_mean:.4f}")
        print(f"   Difference: {diff:+.4f} ({percent_change:+.2f}%)")
        
        if abs(percent_change) > 5:
            print(f"   ⚠️  Significant change (>5%)")
        else:
            print(f"   ✅ Similar performance")

if 'w3_w5_similarity' in summary_df.columns:
    print("\n\n📊 W3-W5 Embedding Similarity Analysis:")
    print("─"*100)
    
    avg_sim = summary_df['w3_w5_similarity'].mean()
    min_sim = summary_df['w3_w5_similarity'].min()
    max_sim = summary_df['w3_w5_similarity'].max()
    
    print(f"\nAverage Similarity: {avg_sim:.4f}")
    print(f"Min Similarity:     {min_sim:.4f} ({summary_df.loc[summary_df['w3_w5_similarity'].idxmin(), 'topic']})")
    print(f"Max Similarity:     {max_sim:.4f} ({summary_df.loc[summary_df['w3_w5_similarity'].idxmax(), 'topic']})")
    
    if avg_sim > 0.95:
        print("\n✅ Embeddings are very similar - window size has minimal impact")
    elif avg_sim > 0.85:
        print("\n⚠️  Embeddings are moderately similar - some differences observed")
    else:
        print("\n❌ Embeddings are quite different - window size has significant impact")

print("\n" + "="*100)
print("ANALYSIS COMPLETE")
print("="*100)

---

## 📋 Summary

### **What This Analysis Does:**

1. **Loads W3 and W5 data** for each topic (one at a time)
2. **Compares**:
   - Data volume (row counts)
   - Topic similarity scores (mean, std)
   - Embedding statistics (norm, sparsity)
   - W3-W5 embedding similarity (cosine)
3. **Generates visualizations**:
   - Individual plots per topic (4 subplots each)
   - Combined summary plot (all topics)
4. **Saves results**:
   - CSV summary table
   - PNG plots

### **Memory-Efficient Strategy:**
- ✅ Processes ONE topic at a time
- ✅ Clears memory after each topic
- ✅ Samples embeddings (1000 rows) for comparison
- ✅ Only stores lightweight summary stats

### **Output Files:**
```
Window_Comparison_Results/
├── w3_vs_w5_summary.csv                    # Summary table
├── SUMMARY_w3_vs_w5_all_topics.png        # Combined plot
├── War_w3_vs_w5_comparison.png            # Individual plots
├── Health_w3_vs_w5_comparison.png
├── Technology_w3_vs_w5_comparison.png
├── Climate_w3_vs_w5_comparison.png
└── Economics_w3_vs_w5_comparison.png
```

### **How to Use:**
1. **Update paths** in Step 1 if your data is in a different location
2. **Run cells sequentially** (Step 1 → 2 → 3 → 4 → 5)
3. **Check output** in `Window_Comparison_Results/` folder

### **What to Look For:**
- **High W3-W5 similarity (>0.9)** → Window size doesn't matter much
- **Low similarity (<0.8)** → Window size significantly affects embeddings
- **Topic score differences** → Which window performs better for each topic

---

# Narrative Shift Detection Pipeline
# Data Preprocessing Notebook

---

## Pipeline Overview

**Stage 0:** Raw Dataset - Define paths and libraries  
**Stage 1:** Sentence Segmentation - Break articles into sentences  
**Stage 2:** Context-Aware Sentence Embedding - Encode sentences with neighbors using SBERT  
**Stage 3:** Topic Embedding Construction - Create semantic topic representations  
**Stage 4:** Sentence-Level Topic Weighting - Compute cosine similarity to topics  
**Stage 5:** Topic Filtering and Temporal Ordering - Filter by threshold & sort by date  
**Stage 6:** Same-Date Aggregation - Aggregate same-day articles via mean pooling  
**Stage 7:** Topic-Wise Adaptive Temporal Windowing - Create overlapping temporal windows  
**Stage 8:** Temporal Contrastive Learning (TCL) - Learn shift-sensitive representations  
**Stage 9:** Narrative Shift Scoring - Compute shift magnitude and detect changes  

---

### Key Pipeline Characteristics:
- **Unsupervised:** No labeled data required
- **Entity-Agnostic:** Detects semantic shifts, not entity changes
- **Multi-Topic:** Handles multiple topics simultaneously
- **Fine-Grained:** Operates at sentence level, aggregates temporally
- **Scalable:** Designed for large-scale news corpora

---

---

## Stage 0: Raw Dataset

**Input:** A raw corpus of news articles represented as a table with two columns:
- `Date`: publication date
- `Article`: full article content

**Example Input:**
```
Date: 2022-06-15
Article: "Artificial intelligence is rapidly expanding. Data centers now consume massive 
amounts of electricity. Governments are beginning to regulate AI usage."
```

**Motivation:** 
Articles are long, may contain multiple topics, and often mix several narrative frames. 
Narrative shifts occur at a finer granularity than the article level, motivating further 
decomposition.

**Stage 0 Goal:** 
- Define all data paths available in Data_Files folder
- Import necessary libraries
- Set up configuration (NO data loading yet)

---

In [ ]:
# ============================================================================
# STAGE 0: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# Run this cell first if you want to run Stage 0 independently
# ============================================================================

print("=" * 100)
print("STAGE 0: IMPORTING REQUIRED LIBRARIES")
print("=" * 100)

# Core data manipulation
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Date and time handling
from datetime import datetime, timedelta

# File operations
import json
import csv
import glob

# Display and progress
from IPython.display import display, clear_output
from tqdm import tqdm

print("✅ All Stage 0 libraries imported successfully")
print("\n" + "=" * 100)
print("✅ STAGE 0 LIBRARIES READY - You can now run Stage 0 cells")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 0: RAW DATASET - SETUP AND PATH DEFINITION
# ============================================================================
# Purpose: Define all data paths and import necessary libraries
# Note: NO data loading in this stage - only setup and path configuration
# ============================================================================

print("=" * 100)
print("STAGE 0: RAW DATASET - SETUP AND PATH DEFINITION")
print("=" * 100)

# ----------------------------------------------------------------------------
# 1. Import Required Libraries
# ----------------------------------------------------------------------------
print("\n📦 Importing Libraries...")

# Core data manipulation
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Date and time handling
from datetime import datetime, timedelta

# File operations
import json
import csv
import glob

# Display and progress
from IPython.display import display, clear_output
from tqdm import tqdm

print("✅ Core libraries imported successfully")

# ----------------------------------------------------------------------------
# 2. Define Data Folder Path
# ----------------------------------------------------------------------------
print("\n📁 Defining Data Folder Path...")

# Main data folder containing all CSV files
DATA_FILES_FOLDER = Path('/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_0')

# Verify folder exists
if DATA_FILES_FOLDER.exists():
    print(f"✅ Data folder found: {DATA_FILES_FOLDER.absolute()}")
else:
    print(f"❌ Data folder not found: {DATA_FILES_FOLDER.absolute()}")
    print("   Please ensure the Data_Files folder exists in the same directory as this notebook")

# ----------------------------------------------------------------------------
# 3. Discover and Define All Data File Paths
# ----------------------------------------------------------------------------
print("\n🔍 Discovering Data Files...")

# Get all CSV files in the Data_Files folder
data_file_paths = sorted(DATA_FILES_FOLDER.glob('Data_*.csv'))

# Extract file numbers for sorting (Data_1.csv -> 1, Data_10.csv -> 10, etc.)
def extract_file_number(file_path):
    """Extract numeric part from filename like 'Data_123.csv' -> 123"""
    try:
        return int(file_path.stem.split('_')[1])
    except:
        return 0

# Sort files by number (Data_1, Data_2, ..., Data_347)
data_file_paths = sorted(data_file_paths, key=extract_file_number)

# Convert to list of Path objects
DATA_FILE_PATHS = data_file_paths

print(f"✅ Discovered {len(DATA_FILE_PATHS)} data files")

# ----------------------------------------------------------------------------
# 4. Display File Path Information
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("📊 DATA FILE INVENTORY")
print("=" * 100)

print(f"\n📈 Total Files: {len(DATA_FILE_PATHS)}")
print(f"📂 Location: {DATA_FILES_FOLDER.absolute()}")

# Show first 10 and last 10 files
print(f"\n📋 First 10 Files:")
for i, file_path in enumerate(DATA_FILE_PATHS[:10], 1):
    print(f"   {i:3d}. {file_path.name}")

if len(DATA_FILE_PATHS) > 20:
    print(f"\n   ... ({len(DATA_FILE_PATHS) - 20} files omitted) ...")

if len(DATA_FILE_PATHS) > 10:
    print(f"\n📋 Last 10 Files:")
    for i, file_path in enumerate(DATA_FILE_PATHS[-10:], len(DATA_FILE_PATHS) - 9):
        print(f"   {i:3d}. {file_path.name}")

# ----------------------------------------------------------------------------
# 5. Configuration Settings
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("⚙️  CONFIGURATION SETTINGS")
print("=" * 100)

# Expected columns in each CSV file
EXPECTED_COLUMNS = ['Date', 'Article', 'Source']

# Data configuration
CONFIG = {
    'data_folder': str(DATA_FILES_FOLDER),
    'total_files': len(DATA_FILE_PATHS),
    'expected_columns': EXPECTED_COLUMNS,
    'articles_per_file': 10000,  # Expected number of articles per file
    'date_column': 'Date',
    'article_column': 'Article',
    'source_column': 'Source'
}

print(f"\n📊 Configuration:")
for key, value in CONFIG.items():
    print(f"   {key}: {value}")

# ----------------------------------------------------------------------------
# 6. Create Helper Functions for File Access
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("🔧 HELPER FUNCTIONS DEFINED")
print("=" * 100)

def get_file_path(file_number):
    """
    Get the path for a specific data file by number.
    
    Args:
        file_number (int): File number (1 to total_files)
    
    Returns:
        Path: File path or None if not found
    """
    if 1 <= file_number <= len(DATA_FILE_PATHS):
        return DATA_FILE_PATHS[file_number - 1]
    else:
        print(f"⚠️  Invalid file number: {file_number} (valid range: 1-{len(DATA_FILE_PATHS)})")
        return None

def get_all_file_paths():
    """
    Get all data file paths.
    
    Returns:
        list: List of all file paths
    """
    return DATA_FILE_PATHS

def get_file_count():
    """
    Get total number of data files.
    
    Returns:
        int: Number of files
    """
    return len(DATA_FILE_PATHS)

def estimate_total_articles():
    """
    Estimate total number of articles across all files.
    
    Returns:
        int: Estimated total articles
    """
    return len(DATA_FILE_PATHS) * CONFIG['articles_per_file']

print("✅ Helper functions defined:")
print("   - get_file_path(file_number): Get path for specific file")
print("   - get_all_file_paths(): Get all file paths")
print("   - get_file_count(): Get total number of files")
print("   - estimate_total_articles(): Estimate total articles")

# ----------------------------------------------------------------------------
# 7. Quick Statistics (without loading data)
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("📈 ESTIMATED STATISTICS")
print("=" * 100)

estimated_articles = estimate_total_articles()

print(f"\n📊 Raw Dataset Overview (estimated):")
print(f"   Total Files: {get_file_count():,}")
print(f"   Estimated Articles: ~{estimated_articles:,}")
print(f"   Articles per File: ~{CONFIG['articles_per_file']:,}")
print(f"   Expected Columns: {', '.join(CONFIG['expected_columns'])}")

# ----------------------------------------------------------------------------
# 8. Path Export for Next Stages
# ----------------------------------------------------------------------------
print("\n" + "=" * 100)
print("✅ STAGE 0 COMPLETE - PATHS AND CONFIGURATION READY")
print("=" * 100)

print(f"\n💡 Available Variables:")
print(f"   - DATA_FILE_PATHS: List of {len(DATA_FILE_PATHS)} file paths")
print(f"   - DATA_FILES_FOLDER: Path object pointing to Data_Files folder")
print(f"   - CONFIG: Configuration dictionary")
print(f"   - Helper functions: get_file_path(), get_all_file_paths(), etc.")

print(f"\n📌 Next Stage: Load and process articles from these files")
print(f"=" * 100)

---

## Stage 1: Sentence Segmentation

**Goal:** Break each article into sentences while preserving temporal order and context

**Input:** Raw articles from Data_Files (Date, Article, Source)

**Output Format:**
- `sentence_id`: Unique identifier for each sentence
- `article_id`: Original article identifier
- `date`: Publication date (preserved from article)
- `source`: Article source
- `previous_sentence_1`: The sentence 2 positions before (or "" if not available)
- `previous_sentence_2`: The sentence 1 position before (or "" if first)
- `main_sentence`: Current sentence
- `next_sentence_1`: The sentence 1 position after (or "" if last)
- `next_sentence_2`: The sentence 2 positions after (or "" if not available)

**Context Window:** Size 5 (2 previous + 1 main + 2 next sentences)

**Why this structure:**
- Preserves sentence order within articles
- Provides wider context (2 sentences before and after) for embedding in Stage 2
- Enables richer context-aware semantic encoding
- Window size of 5 captures more surrounding narrative context

**Processing Strategy:**
- Multi-threaded file processing (one file at a time)
- Each file processed completely before moving to next
- Output saved to `Processed_Data/Stage_1/` folder

---

In [ ]:
# ============================================================================
# STAGE 1: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# Run this cell first if you want to run Stage 1 independently
# ============================================================================

print("=" * 100)
print("STAGE 1: IMPORTING REQUIRED LIBRARIES")
print("=" * 100)

# Core libraries from Stage 0
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
import json
import csv
import glob
from IPython.display import display, clear_output
from tqdm import tqdm

# Stage 1 specific libraries
import nltk
from nltk.tokenize import sent_tokenize
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

print("✅ All libraries imported successfully")

# Define paths if not already defined
if 'DATA_FILES_FOLDER' not in dir():
    DATA_FILES_FOLDER = Path('Data_Files')
    print(f"✅ DATA_FILES_FOLDER defined: {DATA_FILES_FOLDER.absolute()}")

if 'STAGE_1_OUTPUT_FOLDER' not in dir():
    STAGE_1_OUTPUT_FOLDER = Path('Processed_Data/Stage_1')
    print(f"✅ STAGE_1_OUTPUT_FOLDER defined: {STAGE_1_OUTPUT_FOLDER.absolute()}")

# Helper functions if not already defined
if 'get_all_file_paths' not in dir():
    def get_all_file_paths():
        """Get all data file paths"""
        data_file_paths = sorted(DATA_FILES_FOLDER.glob('Data_*.csv'))
        def extract_file_number(file_path):
            try:
                return int(file_path.stem.split('_')[1])
            except:
                return 0
        return sorted(data_file_paths, key=extract_file_number)
    print("✅ Helper function get_all_file_paths() defined")

print("\n" + "=" * 100)
print("✅ STAGE 1 LIBRARIES READY - You can now run Stage 1 cells")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 1: SENTENCE SEGMENTATION - SETUP
# ============================================================================
# Import NLP library for sentence tokenization
# ============================================================================

print("=" * 100)
print("STAGE 1: SENTENCE SEGMENTATION - SETUP")
print("=" * 100)

# Import sentence tokenization library
try:
    import nltk
    from nltk.tokenize import sent_tokenize
    print("✅ NLTK already installed")
except ImportError:
    print("📥 Installing NLTK...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nltk'])
    import nltk
    from nltk.tokenize import sent_tokenize
    print("✅ NLTK installed successfully")

# Download punkt tokenizer if not already downloaded
try:
    nltk.data.find('tokenizers/punkt')
    print("✅ Punkt tokenizer already available")
except LookupError:
    print("📥 Downloading punkt tokenizer...")
    nltk.download('punkt', quiet=True)
    print("✅ Punkt tokenizer downloaded")

# Import threading libraries
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

print("\n✅ All libraries for Stage 1 ready")

# ----------------------------------------------------------------------------
# Configure Output Folder
# ----------------------------------------------------------------------------
print("\n📁 Configuring Output Folder...")

# Create output folder for Stage 1
STAGE_1_OUTPUT_FOLDER = Path('/home/hp/SEM2/INLP/Naretve_Shift/Processed_Data/Stage_1')
STAGE_1_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"✅ Output folder: {STAGE_1_OUTPUT_FOLDER.absolute()}")

# Configuration for Stage 1
STAGE_1_CONFIG = {
    'output_folder': str(STAGE_1_OUTPUT_FOLDER),
    'output_file_prefix': 'Data_s1_',
    'columns': ['sentence_id', 'article_id', 'date', 'source', 
                'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 
                'next_sentence_1', 'next_sentence_2'],
    'window_size': 5,  # Context window: 2 previous + 1 main + 2 next
    'num_threads': 8,  # Number of parallel threads for file processing
    'chunk_size': 1000  # Process articles in chunks for memory efficiency
}

print(f"\n⚙️  Stage 1 Configuration:")
for key, value in STAGE_1_CONFIG.items():
    print(f"   {key}: {value}")

print("\n" + "=" * 100)
print("✅ STAGE 1 SETUP COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 1: SENTENCE SEGMENTATION - PROCESSING FUNCTIONS
# ============================================================================
# Define functions for sentence segmentation with context
# ============================================================================

def segment_article_into_sentences(article_text, article_id, date, source, window_size=5):
    """
    Segment a single article into sentences with context window (2 previous + main + 2 next).
    
    Args:
        article_text (str): Full article text
        article_id (str): Unique article identifier
        date (str): Publication date
        source (str): Article source
        window_size (int): Total context window size (default: 5 = 2 previous + 1 main + 2 next)
    
    Returns:
        list: List of sentence dictionaries with context
    """
    # Tokenize article into sentences
    sentences = sent_tokenize(str(article_text))
    
    # Calculate context size (sentences before and after main sentence)
    context_before = (window_size - 1) // 2  # 2 sentences before
    context_after = window_size - 1 - context_before  # 2 sentences after
    
    # Create sentence records with context
    sentence_records = []
    
    for i, main_sentence in enumerate(sentences):
        # Skip empty sentences
        if not main_sentence or str(main_sentence).strip() == "":
            continue
        
        # Get 2 previous sentences (or fewer if at beginning)
        previous_sentence_1 = sentences[i - 2] if i >= 2 else ""
        previous_sentence_2 = sentences[i - 1] if i >= 1 else ""
        
        # Get 2 next sentences (or fewer if at end)
        next_sentence_1 = sentences[i + 1] if i < len(sentences) - 1 else ""
        next_sentence_2 = sentences[i + 2] if i < len(sentences) - 2 else ""
        
        # Create unique sentence ID (use original index i+1 to maintain sequence)
        sentence_id = f"{article_id}_s{len(sentence_records) + 1}"
        
        # Create sentence record - ensure empty strings instead of None/NaN
        record = {
            'sentence_id': sentence_id,
            'article_id': article_id,
            'date': date,
            'source': source,
            'previous_sentence_1': previous_sentence_1 if previous_sentence_1 else "",
            'previous_sentence_2': previous_sentence_2 if previous_sentence_2 else "",
            'main_sentence': main_sentence.strip(),
            'next_sentence_1': next_sentence_1 if next_sentence_1 else "",
            'next_sentence_2': next_sentence_2 if next_sentence_2 else ""
        }
        
        sentence_records.append(record)
    
    return sentence_records


def process_single_file(file_path, file_number, total_files):
    """
    Process a single data file and perform sentence segmentation.
    
    Args:
        file_path (Path): Path to the CSV file
        file_number (int): File number (for tracking)
        total_files (int): Total number of files
    
    Returns:
        tuple: (file_number, sentence_count, output_file_path, success)
    """
    try:
        start_time = time.time()
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Validate required columns
        required_cols = ['Date', 'Article', 'Source']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  Skipping {file_path.name}: Missing required columns")
            return (file_number, 0, None, False, 0)
        
        # Process each article
        all_sentences = []
        
        for idx, row in df.iterrows():
            article_id = f"f{file_number}_a{idx + 1}"
            date = row['Date']
            article_text = row['Article']
            source = row['Source']
            
            # Skip if article text is NaN or empty
            if pd.isna(article_text) or str(article_text).strip() == "":
                continue
            
            # Segment article into sentences
            sentences = segment_article_into_sentences(
                article_text, article_id, date, source
            )
            
            all_sentences.extend(sentences)
        
        # Create DataFrame from sentences
        sentences_df = pd.DataFrame(all_sentences)
        
        # Replace any remaining NaN values with empty strings
        sentences_df = sentences_df.fillna("")
        
        # Define output file path
        output_file = STAGE_1_OUTPUT_FOLDER / f"Data_s1_{file_number}.csv"
        
        # Save to CSV
        sentences_df.to_csv(output_file, index=False)
        
        elapsed_time = time.time() - start_time
        
        return (file_number, len(sentences_df), output_file, True, elapsed_time)
        
    except Exception as e:
        print(f"\n❌ Error processing {file_path.name}: {e}")
        return (file_number, 0, None, False, 0)


print("✅ Sentence segmentation functions defined:")
print("   - segment_article_into_sentences(): Segment one article")
print("   - process_single_file(): Process entire file")

In [ ]:
# ============================================================================
# STAGE 1: SENTENCE SEGMENTATION - MAIN PROCESSING
# ============================================================================
# Process all files with multi-threading for improved speed
# ============================================================================

print("=" * 100)
print("STAGE 1: SENTENCE SEGMENTATION - MAIN PROCESSING")
print("=" * 100)

# Get all file paths
all_files = get_all_file_paths()
total_files = len(all_files)
num_threads = STAGE_1_CONFIG['num_threads']

print(f"\n📊 Processing Overview:")
print(f"   Total files to process: {total_files:,}")
print(f"   Output folder: {STAGE_1_OUTPUT_FOLDER.absolute()}")
print(f"   Number of threads: {num_threads}")
print(f"   Processing mode: Multi-threaded (parallel processing)")

# Initialize tracking variables
processed_files = 0
total_sentences = 0
successful_files = []
failed_files = []
start_time_overall = time.time()

# Thread tracking - stores currently processing files per thread
thread_status = {}  # {thread_id: {'file_number': X, 'file_name': 'Data_X.csv', 'start_time': time}}
completed_files = []  # Track recently completed files

# Thread-safe lock for updating progress
progress_lock = threading.Lock()

def mark_thread_start(thread_id, file_number, file_name):
    """Mark that a thread has started processing a file"""
    with progress_lock:
        thread_status[thread_id] = {
            'file_number': file_number,
            'file_name': file_name,
            'start_time': time.time()
        }

def mark_thread_complete(thread_id):
    """Mark that a thread has completed processing"""
    with progress_lock:
        if thread_id in thread_status:
            del thread_status[thread_id]

def update_progress(file_number, file_name, sentence_count, output_file, success, elapsed):
    """Thread-safe progress update"""
    global processed_files, total_sentences
    
    with progress_lock:
        if success:
            processed_files += 1
            total_sentences += sentence_count
            file_info = {
                'file_number': file_number,
                'file_name': file_name,
                'sentences': sentence_count,
                'output': output_file.name,
                'time': elapsed
            }
            successful_files.append(file_info)
            completed_files.append(file_info)
            
            # Keep only last 10 completed files for display
            if len(completed_files) > 10:
                completed_files.pop(0)
            
            # Calculate statistics
            avg_time_per_file = (time.time() - start_time_overall) / processed_files
            estimated_remaining_time = avg_time_per_file * (total_files - processed_files)
            
            # Clear and display progress
            clear_output(wait=True)
            print("=" * 120)
            print(f"🚀 PROCESSING IN PROGRESS - {num_threads} THREADS ACTIVE")
            print("=" * 120)
            
            # Overall Progress bar
            progress_pct = (processed_files / total_files) * 100
            bar_length = 60
            filled_length = int(bar_length * processed_files / total_files)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)
            
            print(f"\n📊 Overall Progress:")
            print(f"   [{bar}] {progress_pct:.1f}%")
            print(f"   Files: {processed_files}/{total_files} | Sentences: {total_sentences:,}")
            print(f"   Time Elapsed: {(time.time() - start_time_overall)/60:.1f} min | ETA: {estimated_remaining_time/60:.1f} min")
            print(f"   Avg Speed: {avg_time_per_file:.2f}s/file | Processing Rate: {total_files/(time.time() - start_time_overall)*60:.1f} files/min")
            
            # Thread-by-thread status
            print(f"\n🔄 Active Threads ({len(thread_status)}/{num_threads}):")
            print("-" * 120)
            if thread_status:
                # Sort threads by thread ID for consistent display
                sorted_threads = sorted(thread_status.items(), key=lambda x: x[0])
                for thread_id, status in sorted_threads:
                    elapsed_thread = time.time() - status['start_time']
                    # Create mini progress bar for thread
                    thread_bar_length = 15
                    # Estimate progress based on average file time
                    if processed_files > 0:
                        thread_progress = min(1.0, elapsed_thread / avg_time_per_file)
                    else:
                        thread_progress = 0.3  # Default for first files
                    thread_filled = int(thread_bar_length * thread_progress)
                    thread_bar = '█' * thread_filled + '░' * (thread_bar_length - thread_filled)
                    
                    print(f"   Thread {thread_id:2d}: [{thread_bar}] Processing File {status['file_number']:3d} "
                          f"({status['file_name']:<20}) - {elapsed_thread:.1f}s")
            else:
                print("   All threads idle")
            
            # Show recently completed files
            print(f"\n✅ Recently Completed Files:")
            print("-" * 120)
            for file_info in completed_files[-8:]:  # Show last 8
                print(f"   File {file_info['file_number']:3d}: {file_info['output']:<25} | "
                      f"Sentences: {file_info['sentences']:>7,} | Time: {file_info['time']:>5.2f}s")
            
        else:
            failed_files.append({
                'file_number': file_number,
                'file_name': file_name
            })
            print(f"\n❌ Failed: File {file_number} - {file_name}")

print(f"\n🚀 Starting multi-threaded sentence segmentation...")
print("=" * 120)

# Initial progress display
clear_output(wait=True)
print("=" * 120)
print(f"🚀 STARTING PROCESSING - {num_threads} THREADS INITIALIZING")
print("=" * 120)
print(f"\n📊 Ready to process {total_files:,} files")
print(f"   Output: {STAGE_1_OUTPUT_FOLDER.absolute()}")

# Wrapper function to track thread activity
def process_file_with_tracking(file_path, file_idx, total_files, thread_id):
    """Wrapper to track thread activity during processing"""
    # Mark thread as started BEFORE processing
    mark_thread_start(thread_id, file_idx, file_path.name)
    
    try:
        # Process the file
        result = process_single_file(file_path, file_idx, total_files)
        return result
    finally:
        # Mark thread as complete AFTER processing (but this happens fast)
        mark_thread_complete(thread_id)

# Process files using ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    # Submit files in batches to control thread assignment
    futures = []
    future_to_info = {}
    
    for file_idx, file_path in enumerate(all_files, 1):
        thread_id = (file_idx - 1) % num_threads  # Round-robin thread assignment
        future = executor.submit(process_file_with_tracking, file_path, file_idx, total_files, thread_id)
        futures.append(future)
        future_to_info[future] = (file_idx, file_path, thread_id)
    
    # Process completed tasks as they finish
    for future in as_completed(futures):
        file_idx, file_path, thread_id = future_to_info[future]
        try:
            result = future.result()
            file_number, sentence_count, output_file, success, elapsed = result
            update_progress(file_number, file_path.name, sentence_count, output_file, success, elapsed)
        except Exception as e:
            mark_thread_complete(thread_id)  # Ensure thread is marked complete on error
            print(f"\n❌ Exception processing file {file_idx}: {e}")
            update_progress(file_idx, file_path.name, 0, None, False, 0)

# Calculate final statistics
elapsed_time_overall = time.time() - start_time_overall

# Clear and show final summary
clear_output(wait=True)

print("\n" + "=" * 100)
print("📈 STAGE 1 PROCESSING COMPLETE")
print("=" * 100)

print(f"\n✅ Summary:")
print(f"   Files processed successfully: {len(successful_files):,}/{total_files:,}")
print(f"   Files failed: {len(failed_files):,}")
print(f"   Total sentences generated: {total_sentences:,}")
print(f"   Total processing time: {elapsed_time_overall/60:.2f} minutes")
if len(successful_files) > 0:
    print(f"   Average time per file: {elapsed_time_overall/len(successful_files):.2f} seconds")
    print(f"   Average sentences per file: {total_sentences/len(successful_files):.0f}")
print(f"   Processing speed: {total_files/(elapsed_time_overall/60):.1f} files/minute")

# Show sample of successful files
if successful_files:
    print(f"\n📋 Sample of Processed Files (first 10):")
    print("-" * 100)
    print(f"{'File #':<10} {'Sentences':<12} {'Output File':<30} {'Time (s)':<10}")
    print("-" * 100)
    # Sort by file number for display
    sorted_files = sorted(successful_files, key=lambda x: x['file_number'])
    for file_info in sorted_files[:10]:
        print(f"{file_info['file_number']:<10} {file_info['sentences']:<12,} "
              f"{file_info['output']:<30} {file_info['time']:<10.2f}")
    
    if len(successful_files) > 10:
        print(f"\n   ... ({len(successful_files) - 10} more files)")

# Show failed files if any
if failed_files:
    print(f"\n⚠️  Failed Files ({len(failed_files)}):")
    for file_info in sorted(failed_files, key=lambda x: x['file_number']):
        print(f"   {file_info['file_number']}. {file_info['file_name']}")

# Save processing summary
summary_file = STAGE_1_OUTPUT_FOLDER / 'processing_summary.json'
summary_data = {
    'total_files': total_files,
    'successful_files': len(successful_files),
    'failed_files': len(failed_files),
    'total_sentences': total_sentences,
    'processing_time_minutes': elapsed_time_overall / 60,
    'average_time_per_file_seconds': elapsed_time_overall / len(successful_files) if len(successful_files) > 0 else 0,
    'processing_speed_files_per_minute': total_files / (elapsed_time_overall / 60),
    'num_threads': num_threads,
    'output_folder': str(STAGE_1_OUTPUT_FOLDER),
    'timestamp': datetime.now().isoformat()
}

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\n💾 Processing summary saved: {summary_file.name}")

print("\n" + "=" * 100)
print("🎉 STAGE 1 COMPLETE!")
print("=" * 100)
print(f"\n💡 Next Stage: Context-Aware Sentence Embedding (Stage 2)")
print(f"   Input: {len(successful_files):,} files with {total_sentences:,} sentences")
print(f"   Location: {STAGE_1_OUTPUT_FOLDER.absolute()}")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 1: VERIFICATION AND SAMPLE DISPLAY
# ============================================================================
# Verify output and display sample sentences
# ============================================================================

print("=" * 100)
print("STAGE 1: VERIFICATION - SAMPLE OUTPUT")
print("=" * 100)

# Get list of output files
output_files = sorted(STAGE_1_OUTPUT_FOLDER.glob('Data_s1_*.csv'))

if len(output_files) > 0:
    print(f"\n✅ Found {len(output_files):,} output files")
    
    # Load first file as sample
    sample_file = output_files[0]
    sample_df = pd.read_csv(sample_file)
    
    print(f"\n📊 Sample Data from: {sample_file.name}")
    print(f"   Total sentences in file: {len(sample_df):,}")
    print(f"   Columns: {list(sample_df.columns)}")
    
    # Display first 5 sentences
    print(f"\n📋 First 5 Sentences (showing window structure):")
    print("=" * 100)
    
    for idx in range(min(5, len(sample_df))):
        row = sample_df.iloc[idx]
        
        # Handle NaN values - convert to string safely (window size 5)
        prev_sent_1 = str(row['previous_sentence_1']) if pd.notna(row['previous_sentence_1']) else ""
        prev_sent_2 = str(row['previous_sentence_2']) if pd.notna(row['previous_sentence_2']) else ""
        main_sent = str(row['main_sentence']) if pd.notna(row['main_sentence']) else ""
        next_sent_1 = str(row['next_sentence_1']) if pd.notna(row['next_sentence_1']) else ""
        next_sent_2 = str(row['next_sentence_2']) if pd.notna(row['next_sentence_2']) else ""
        
        print(f"\n🔹 Sentence {idx + 1}:")
        print(f"   Sentence ID: {row['sentence_id']}")
        print(f"   Article ID: {row['article_id']}")
        print(f"   Date: {row['date']}")
        print(f"   Source: {row['source']}")
        print(f"   Previous-1 (i-2): \"{prev_sent_1[:50]}{'...' if len(prev_sent_1) > 50 else ''}\"")
        print(f"   Previous-2 (i-1): \"{prev_sent_2[:50]}{'...' if len(prev_sent_2) > 50 else ''}\"")
        print(f"   Main (i): \"{main_sent[:60]}{'...' if len(main_sent) > 60 else ''}\"")
        print(f"   Next-1 (i+1): \"{next_sent_1[:50]}{'...' if len(next_sent_1) > 50 else ''}\"")
        print(f"   Next-2 (i+2): \"{next_sent_2[:50]}{'...' if len(next_sent_2) > 50 else ''}\"")
    
    # Statistics
    print(f"\n\n📊 Statistics for Sample File:")
    print("-" * 100)
    
    # Count unique articles
    unique_articles = sample_df['article_id'].nunique()
    avg_sentences_per_article = len(sample_df) / unique_articles
    
    print(f"   Unique articles: {unique_articles:,}")
    print(f"   Average sentences per article: {avg_sentences_per_article:.1f}")
    
    # Count sentences with/without context (handle NaN) - window size 5
    has_prev_1 = sample_df['previous_sentence_1'].notna() & (sample_df['previous_sentence_1'] != "")
    has_prev_2 = sample_df['previous_sentence_2'].notna() & (sample_df['previous_sentence_2'] != "")
    has_next_1 = sample_df['next_sentence_1'].notna() & (sample_df['next_sentence_1'] != "")
    has_next_2 = sample_df['next_sentence_2'].notna() & (sample_df['next_sentence_2'] != "")
    
    has_full_window = has_prev_1 & has_prev_2 & has_next_1 & has_next_2
    has_any_prev = has_prev_1 | has_prev_2
    has_any_next = has_next_1 | has_next_2
    
    print(f"\n   Context Window Statistics (Window Size = 5):")
    print(f"      Sentences with full window (2+1+2): {has_full_window.sum():,} ({has_full_window.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with any previous context: {has_any_prev.sum():,} ({has_any_prev.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with any next context: {has_any_next.sum():,} ({has_any_next.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with prev-1 (i-2): {has_prev_1.sum():,} ({has_prev_1.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with prev-2 (i-1): {has_prev_2.sum():,} ({has_prev_2.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with next-1 (i+1): {has_next_1.sum():,} ({has_next_1.sum()/len(sample_df)*100:.1f}%)")
    print(f"      Sentences with next-2 (i+2): {has_next_2.sum():,} ({has_next_2.sum()/len(sample_df)*100:.1f}%)")
    
    # Display DataFrame info
    print(f"\n\n📊 DataFrame Info:")
    print("-" * 100)
    sample_df.info()
    
    # Display full sample
    print(f"\n\n📋 Full Sample (first 10 rows):")
    print("=" * 100)
    display(sample_df.head(10))
    
else:
    print("\n⚠️  No output files found. Please run the processing cell first.")

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE")
print("=" * 100)

---

## Stage 2: Context-Aware Sentence Embedding

**Problem:** Isolated sentences can be semantically ambiguous. For example, a sentence about energy consumption may appear climate-related unless contextualized.

**Solution:** Each sentence is embedded together with its immediate neighbors using Sentence-BERT (SBERT) with two different window sizes.

**Two Window Sizes:**
1. **Window 3 (w3):** 1 previous + main + 1 next sentence
2. **Window 5 (w5):** 2 previous + main + 2 next sentences

**Method:** For a sentence `si`, we construct two contextual inputs:
- **w3:** `(si-1, si, si+1)` - Immediate context
- **w5:** `(si-2, si-1, si, si+1, si+2)` - Wider context

**Example (for sentence s2):**
- **w3:** `("AI is rapidly expanding.", "Data centers consume electricity.", "Governments regulate AI.")`
- **w5:** `("", "AI is rapidly expanding.", "Data centers consume electricity.", "Governments regulate AI.", "")`

**Input:** Stage 1 output (Data_s1_X.csv) with columns:
- sentence_id, article_id, date, source, previous_sentence_1, previous_sentence_2, main_sentence, next_sentence_1, next_sentence_2

**Output Format:**
- All Stage 1 columns PLUS:
- `w3_embedding`: 768-dimensional SBERT embedding with window size 3
- `w5_embedding`: 768-dimensional SBERT embedding with window size 5

**Processing Strategy:**
- Multi-threaded file processing (2 threads for CPU)
- Use SBERT model: `all-mpnet-base-v2` (768-dim, high quality)
- Each contextual input encoded as: `[prev] [SEP] [main] [SEP] [next]`
- Output saved to `Processed_Data/Stage_2/Data_s2_X.csv`

**Why Two Embeddings:**
- w3: Captures immediate local context (faster, more focused)
- w5: Captures broader narrative context (richer, more comprehensive)
- Allows comparison of different context window effects on shift detection

---

In [1]:
# ============================================================================
# STAGE 2: IMPORT ALL REQUIRED LIBRARIES
# ============================================================================
# Run this cell first if you want to run Stage 2 independently
# ============================================================================

print("=" * 100)
print("STAGE 2: IMPORTING REQUIRED LIBRARIES")
print("=" * 100)

# Core libraries from Stage 0
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
import json
import csv
import glob
from IPython.display import display, clear_output
from tqdm import tqdm

# Threading libraries
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Stage 2 specific libraries
from sentence_transformers import SentenceTransformer

print("✅ All libraries imported successfully")

# Define paths if not already defined
if 'STAGE_1_OUTPUT_FOLDER' not in dir():
    STAGE_1_OUTPUT_FOLDER = Path('Processed_Data/Stage_1')
    print(f"✅ STAGE_1_OUTPUT_FOLDER defined: {STAGE_1_OUTPUT_FOLDER.absolute()}")

if 'STAGE_2_OUTPUT_FOLDER' not in dir():
    STAGE_2_OUTPUT_FOLDER = Path('Processed_Data/Stage_2')
    print(f"✅ STAGE_2_OUTPUT_FOLDER defined: {STAGE_2_OUTPUT_FOLDER.absolute()}")

# Load SBERT model if not already loaded
if 'sbert_model' not in dir():
    print("\n🤖 Loading SBERT model...")
    print("   Device: CPU (limited to 2 cores)")
    
    # Limit to 2 CPU cores
    import os
    os.environ["OMP_NUM_THREADS"] = "2"
    os.environ["MKL_NUM_THREADS"] = "2"
    os.environ["OPENBLAS_NUM_THREADS"] = "2"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
    os.environ["NUMEXPR_NUM_THREADS"] = "2"
    
    sbert_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')
    USE_GPU = False
    BATCH_SIZE = 64
    
    print(f"✅ SBERT model loaded on CPU (2 cores only)")
    print(f"   Embedding dim: {sbert_model.get_sentence_embedding_dimension()}")

# Define Stage 2 config if not already defined
if 'STAGE_2_CONFIG' not in dir():
    STAGE_2_CONFIG = {
        'input_folder': str(STAGE_1_OUTPUT_FOLDER),
        'output_folder': str(STAGE_2_OUTPUT_FOLDER),
        'output_file_prefix': 'Data_s2_',
        'model_name': 'all-mpnet-base-v2',
        'embedding_dim': sbert_model.get_sentence_embedding_dimension(),
        'use_gpu': False,
        'num_threads': 2,  # Only 2 threads for background processing
        'batch_size': 64
    }
    print(f"✅ STAGE_2_CONFIG defined (2 cores only)")

print("\n" + "=" * 100)
print("✅ STAGE 2 LIBRARIES READY - You can now run Stage 2 cells")
print("=" * 100)

STAGE 2: IMPORTING REQUIRED LIBRARIES
✅ All libraries imported successfully
✅ STAGE_1_OUTPUT_FOLDER defined: /home/hp/SEM2/INLP/Naretve_Shift/Pre_Processin/Processed_Data/Stage_1
✅ STAGE_2_OUTPUT_FOLDER defined: /home/hp/SEM2/INLP/Naretve_Shift/Pre_Processin/Processed_Data/Stage_2

🤖 Loading SBERT model...
   Device: CPU (limited to 2 cores)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 340.53it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SBERT model loaded on CPU (2 cores only)
   Embedding dim: 768
✅ STAGE_2_CONFIG defined (2 cores only)

✅ STAGE 2 LIBRARIES READY - You can now run Stage 2 cells


In [ ]:
# ============================================================================
# STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - SETUP
# ============================================================================
# Install and import Sentence-BERT for semantic embeddings
# ============================================================================

print("=" * 100)
print("STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - SETUP")
print("=" * 100)

# Import sentence transformers library
try:
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers already installed")
except ImportError:
    print("📥 Installing sentence-transformers...")
    import subprocess
    import sys
    import pathlib
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
    from sentence_transformers import SentenceTransformer
    print("✅ sentence-transformers installed successfully")

print("\n✅ All libraries for Stage 2 ready")

# ----------------------------------------------------------------------------
# Load SBERT Model (CPU Mode)
# ----------------------------------------------------------------------------
print("\n🤖 Loading Sentence-BERT Model...")
print("   Model: all-mpnet-base-v2 (768-dimensional embeddings)")
print("   This may take a moment on first run (downloading model ~420MB)...")
print("   Device: CPU (forced for stability)")

# Force CPU usage
sbert_model = SentenceTransformer('all-mpnet-base-v2', device='cpu')

# Limit to 2 CPU cores for background processing
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"

print("✅ SBERT model loaded successfully on CPU")
print(f"   CPU cores limited to: 2 (for background processing)")
print(f"   Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

# Set configuration for CPU
USE_GPU = False
BATCH_SIZE = 64  # Optimal batch size for CPU

# ----------------------------------------------------------------------------
# Configure Output Folder
# ----------------------------------------------------------------------------
print("\n📁 Configuring Output Folder...")

# Create output folder for Stage 2
STAGE_2_OUTPUT_FOLDER = Path('Processed_Data/Stage_2')
STAGE_2_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"✅ Output folder: {STAGE_2_OUTPUT_FOLDER.absolute()}")

# Configuration for Stage 2
STAGE_2_CONFIG = {
    'input_folder': str(STAGE_1_OUTPUT_FOLDER),
    'output_folder': str(STAGE_2_OUTPUT_FOLDER),
    'output_file_prefix': 'Data_s2_',
    'model_name': 'all-mpnet-base-v2',
    'embedding_dim': sbert_model.get_sentence_embedding_dimension(),
    'device': 'cpu',
    'use_gpu': False,
    'num_threads': 2,  # Only 2 threads to leave CPU available for other work
    'batch_size': 64   # Optimal batch size for CPU
}

print(f"\n⚙️  Stage 2 Configuration:")
for key, value in STAGE_2_CONFIG.items():
    print(f"   {key}: {value}")

print(f"\n💡 Background Processing Mode:")
print(f"   - CPU cores: Limited to 2 cores")
print(f"   - Threads: 2 (leaves CPU available for other tasks)")
print(f"   - Batch size: 64")
print(f"   - Expected time: ~80-100s per file (~8-10 hours total)")
print(f"   - ✅ You can work on other tasks while this runs!")

print("\n" + "=" * 100)
print("✅ STAGE 2 SETUP COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - PROCESSING FUNCTIONS
# ============================================================================
# Define functions for creating context-aware embeddings with two window sizes
# ============================================================================

def create_contextual_input_w3(prev_sent_i_minus_1, main_sent, next_sent_i_plus_1):
    """
    Create contextual input with window size 3 (1 previous + main + 1 next).
    
    Window 3 uses positions: (i-1, i, i+1)
    - prev_sent_i_minus_1: sentence at position (i-1) = previous_sentence_2 in Stage 1
    - main_sent: sentence at position (i) = main_sentence in Stage 1
    - next_sent_i_plus_1: sentence at position (i+1) = next_sentence_1 in Stage 1
    
    Args:
        prev_sent_i_minus_1 (str): Previous sentence 1 position back (i-1)
        main_sent (str): Main sentence (current, position i)
        next_sent_i_plus_1 (str): Next sentence 1 position ahead (i+1)
    
    Returns:
        str: Concatenated contextual input with [SEP] tokens (window size 3)
    """
    parts = []
    
    # Add previous sentence (i-1) if available
    if prev_sent_i_minus_1 and str(prev_sent_i_minus_1).strip():
        parts.append(str(prev_sent_i_minus_1).strip())
    
    # Always add main sentence (i)
    parts.append(str(main_sent).strip())
    
    # Add next sentence (i+1) if available
    if next_sent_i_plus_1 and str(next_sent_i_plus_1).strip():
        parts.append(str(next_sent_i_plus_1).strip())
    
    return " [SEP] ".join(parts)


def create_contextual_input_w5(prev_sent_i_minus_2, prev_sent_i_minus_1, main_sent, 
                                next_sent_i_plus_1, next_sent_i_plus_2):
    """
    Create contextual input with window size 5 (2 previous + main + 2 next).
    
    Window 5 uses positions: (i-2, i-1, i, i+1, i+2)
    - prev_sent_i_minus_2: sentence at position (i-2) = previous_sentence_1 in Stage 1
    - prev_sent_i_minus_1: sentence at position (i-1) = previous_sentence_2 in Stage 1
    - main_sent: sentence at position (i) = main_sentence in Stage 1
    - next_sent_i_plus_1: sentence at position (i+1) = next_sentence_1 in Stage 1
    - next_sent_i_plus_2: sentence at position (i+2) = next_sentence_2 in Stage 1
    
    Args:
        prev_sent_i_minus_2 (str): Previous sentence 2 positions back (i-2)
        prev_sent_i_minus_1 (str): Previous sentence 1 position back (i-1)
        main_sent (str): Main sentence (current, position i)
        next_sent_i_plus_1 (str): Next sentence 1 position ahead (i+1)
        next_sent_i_plus_2 (str): Next sentence 2 positions ahead (i+2)
    
    Returns:
        str: Concatenated contextual input with [SEP] tokens (window size 5)
    """
    parts = []
    
    # Add previous sentence (i-2) if available
    if prev_sent_i_minus_2 and str(prev_sent_i_minus_2).strip():
        parts.append(str(prev_sent_i_minus_2).strip())
    
    # Add previous sentence (i-1) if available
    if prev_sent_i_minus_1 and str(prev_sent_i_minus_1).strip():
        parts.append(str(prev_sent_i_minus_1).strip())
    
    # Always add main sentence (i)
    parts.append(str(main_sent).strip())
    
    # Add next sentence (i+1) if available
    if next_sent_i_plus_1 and str(next_sent_i_plus_1).strip():
        parts.append(str(next_sent_i_plus_1).strip())
    
    # Add next sentence (i+2) if available
    if next_sent_i_plus_2 and str(next_sent_i_plus_2).strip():
        parts.append(str(next_sent_i_plus_2).strip())
    
    return " [SEP] ".join(parts)


def process_single_file_stage2(input_file_path, file_number, total_files):
    """
    Process a single Stage 1 file and create context-aware embeddings with two window sizes.
    
    Args:
        input_file_path (Path): Path to the Stage 1 CSV file
        file_number (int): File number (for tracking)
        total_files (int): Total number of files
    
    Returns:
        tuple: (file_number, sentence_count, output_file_path, success, elapsed_time)
    """
    try:
        start_time = time.time()
        
        # Read the Stage 1 CSV file
        df = pd.read_csv(input_file_path)
        
        # Validate required columns (updated for window size 5 structure)
        required_cols = ['sentence_id', 'article_id', 'date', 'source', 
                        'previous_sentence_1', 'previous_sentence_2', 'main_sentence', 
                        'next_sentence_1', 'next_sentence_2']
        if not all(col in df.columns for col in required_cols):
            print(f"⚠️  Skipping {input_file_path.name}: Missing required columns")
            return (file_number, 0, None, False, 0)
        
        # Replace NaN with empty strings
        df = df.fillna("")
        
        # Create contextual inputs for both window sizes
        contextual_inputs_w3 = []
        contextual_inputs_w5 = []
        
        for idx, row in df.iterrows():
            # Window 3: Use positions (i-1, i, i+1)
            # - i-1 = previous_sentence_2
            # - i = main_sentence
            # - i+1 = next_sentence_1
            input_w3 = create_contextual_input_w3(
                row['previous_sentence_2'],  # i-1 (prev_sent_i_minus_1)
                row['main_sentence'],         # i (main_sent)
                row['next_sentence_1']        # i+1 (next_sent_i_plus_1)
            )
            contextual_inputs_w3.append(input_w3)
            
            # Window 5: Use positions (i-2, i-1, i, i+1, i+2)
            # - i-2 = previous_sentence_1
            # - i-1 = previous_sentence_2
            # - i = main_sentence
            # - i+1 = next_sentence_1
            # - i+2 = next_sentence_2
            input_w5 = create_contextual_input_w5(
                row['previous_sentence_1'],  # i-2 (prev_sent_i_minus_2)
                row['previous_sentence_2'],  # i-1 (prev_sent_i_minus_1)
                row['main_sentence'],         # i (main_sent)
                row['next_sentence_1'],       # i+1 (next_sent_i_plus_1)
                row['next_sentence_2']        # i+2 (next_sent_i_plus_2)
            )
            contextual_inputs_w5.append(input_w5)
        
        # Generate embeddings in batches for efficiency
        batch_size = STAGE_2_CONFIG['batch_size']
        
        # Generate w3 embeddings
        all_embeddings_w3 = []
        for i in range(0, len(contextual_inputs_w3), batch_size):
            batch = contextual_inputs_w3[i:i + batch_size]
            batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
            all_embeddings_w3.extend(batch_embeddings)
        
        # Generate w5 embeddings
        all_embeddings_w5 = []
        for i in range(0, len(contextual_inputs_w5), batch_size):
            batch = contextual_inputs_w5[i:i + batch_size]
            batch_embeddings = sbert_model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
            all_embeddings_w5.extend(batch_embeddings)
        
        # Convert embeddings lists to numpy arrays
        embeddings_array_w3 = np.array(all_embeddings_w3)
        embeddings_array_w5 = np.array(all_embeddings_w5)
        
        # Add embeddings as new columns (store as comma-separated strings for CSV)
        df['w3_embedding'] = [','.join(map(str, emb)) for emb in embeddings_array_w3]
        df['w5_embedding'] = [','.join(map(str, emb)) for emb in embeddings_array_w5]
        
        # Define output file path
        output_file = STAGE_2_OUTPUT_FOLDER / f"Data_s2_{file_number}.csv"
        
        # Save to CSV
        df.to_csv(output_file, index=False)
        
        elapsed_time = time.time() - start_time
        
        return (file_number, len(df), output_file, True, elapsed_time)
        
    except Exception as e:
        print(f"\n❌ Error processing {input_file_path.name}: {e}")
        import traceback
        traceback.print_exc()
        return (file_number, 0, None, False, 0)


print("✅ Stage 2 processing functions defined:")
print("   - create_contextual_input_w3(): Window size 3 (1 prev + main + 1 next)")
print("   - create_contextual_input_w5(): Window size 5 (2 prev + main + 2 next)")
print("   - process_single_file_stage2(): Generate both w3 and w5 embeddings for file")

In [ ]:
# ============================================================================
# STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - MAIN PROCESSING
# ============================================================================
# Process all Stage 1 files with multi-threading
# ============================================================================

print("=" * 100)
print("STAGE 2: CONTEXT-AWARE SENTENCE EMBEDDING - MAIN PROCESSING")
print("=" * 100)

# Get all Stage 1 output files
stage1_files = sorted(STAGE_1_OUTPUT_FOLDER.glob('Data_s1_*.csv'))
total_files = len(stage1_files)
num_threads = STAGE_2_CONFIG['num_threads']

print(f"\n📊 Processing Overview:")
print(f"   Total files to process: {total_files:,}")
print(f"   Input folder: {STAGE_1_OUTPUT_FOLDER.absolute()}")
print(f"   Output folder: {STAGE_2_OUTPUT_FOLDER.absolute()}")
print(f"   Number of threads: {num_threads}")
print(f"   Batch size: {STAGE_2_CONFIG['batch_size']}")
print(f"   Embedding dimension: {STAGE_2_CONFIG['embedding_dim']}")
print(f"   Window sizes: w3 (1+1+1) and w5 (2+1+2)")
print(f"   Output columns: w3_embedding, w5_embedding")

# Initialize tracking variables
processed_files = 0
total_sentences = 0
successful_files = []
failed_files = []
start_time_overall = time.time()

# Thread tracking
thread_status = {}
completed_files = []

# Thread-safe lock
progress_lock = threading.Lock()

def mark_thread_start_s2(thread_id, file_number, file_name):
    """Mark that a thread has started processing a file"""
    with progress_lock:
        thread_status[thread_id] = {
            'file_number': file_number,
            'file_name': file_name,
            'start_time': time.time()
        }

def mark_thread_complete_s2(thread_id):
    """Mark that a thread has completed processing"""
    with progress_lock:
        if thread_id in thread_status:
            del thread_status[thread_id]

def update_progress_s2(file_number, file_name, sentence_count, output_file, success, elapsed):
    """Thread-safe progress update"""
    global processed_files, total_sentences
    
    with progress_lock:
        if success:
            processed_files += 1
            total_sentences += sentence_count
            file_info = {
                'file_number': file_number,
                'file_name': file_name,
                'sentences': sentence_count,
                'output': output_file.name,
                'time': elapsed
            }
            successful_files.append(file_info)
            completed_files.append(file_info)
            
            if len(completed_files) > 10:
                completed_files.pop(0)
            
            avg_time_per_file = (time.time() - start_time_overall) / processed_files
            estimated_remaining_time = avg_time_per_file * (total_files - processed_files)
            
            # Clear and display progress
            clear_output(wait=True)
            print("=" * 120)
            print(f"🚀 STAGE 2 PROCESSING IN PROGRESS - {num_threads} THREADS ACTIVE")
            print("=" * 120)
            
            # Overall Progress bar
            progress_pct = (processed_files / total_files) * 100
            bar_length = 60
            filled_length = int(bar_length * processed_files / total_files)
            bar = '█' * filled_length + '░' * (bar_length - filled_length)
            
            print(f"\n📊 Overall Progress:")
            print(f"   [{bar}] {progress_pct:.1f}%")
            print(f"   Files: {processed_files}/{total_files} | Sentences: {total_sentences:,}")
            print(f"   Time Elapsed: {(time.time() - start_time_overall)/60:.1f} min | ETA: {estimated_remaining_time/60:.1f} min")
            print(f"   Avg Speed: {avg_time_per_file:.2f}s/file | Processing Rate: {processed_files/(time.time() - start_time_overall)*60:.1f} files/min")
            
            # Thread-by-thread status
            print(f"\n🔄 Active Threads ({len(thread_status)}/{num_threads}):")
            print("-" * 120)
            if thread_status:
                sorted_threads = sorted(thread_status.items(), key=lambda x: x[0])
                for thread_id, status in sorted_threads:
                    elapsed_thread = time.time() - status['start_time']
                    thread_bar_length = 15
                    if processed_files > 0:
                        thread_progress = min(1.0, elapsed_thread / avg_time_per_file)
                    else:
                        thread_progress = 0.3
                    thread_filled = int(thread_bar_length * thread_progress)
                    thread_bar = '█' * thread_filled + '░' * (thread_bar_length - thread_filled)
                    
                    print(f"   Thread {thread_id:2d}: [{thread_bar}] Processing File {status['file_number']:3d} "
                          f"({status['file_name']:<25}) - {elapsed_thread:.1f}s")
            else:
                print("   All threads idle")
            
            # Show recently completed files
            print(f"\n✅ Recently Completed Files:")
            print("-" * 120)
            for file_info in completed_files[-8:]:
                print(f"   File {file_info['file_number']:3d}: {file_info['output']:<25} | "
                      f"Sentences: {file_info['sentences']:>7,} | Time: {file_info['time']:>5.2f}s")
            
        else:
            failed_files.append({
                'file_number': file_number,
                'file_name': file_name
            })
            print(f"\n❌ Failed: File {file_number} - {file_name}")

print(f"\n🚀 Starting multi-threaded embedding generation...")
print("=" * 120)

# Initial progress display
clear_output(wait=True)
print("=" * 120)
print(f"🚀 STAGE 2 STARTING - {num_threads} THREADS INITIALIZING")
print("=" * 120)
print(f"\n📊 Ready to process {total_files:,} files")
print(f"   Creating {STAGE_2_CONFIG['embedding_dim']}-dimensional embeddings...")

# Wrapper function to track thread activity
def process_file_with_tracking_s2(input_file_path, file_idx, total_files, thread_id):
    """Wrapper to track thread activity during processing"""
    mark_thread_start_s2(thread_id, file_idx, input_file_path.name)
    
    try:
        result = process_single_file_stage2(input_file_path, file_idx, total_files)
        return result
    finally:
        mark_thread_complete_s2(thread_id)

# Process files using ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = []
    future_to_info = {}
    
    for file_idx, input_file_path in enumerate(stage1_files, 1):
        thread_id = (file_idx - 1) % num_threads
        future = executor.submit(process_file_with_tracking_s2, input_file_path, file_idx, total_files, thread_id)
        futures.append(future)
        future_to_info[future] = (file_idx, input_file_path, thread_id)
    
    # Process completed tasks as they finish
    for future in as_completed(futures):
        file_idx, input_file_path, thread_id = future_to_info[future]
        try:
            result = future.result()
            file_number, sentence_count, output_file, success, elapsed = result
            update_progress_s2(file_number, input_file_path.name, sentence_count, output_file, success, elapsed)
        except Exception as e:
            mark_thread_complete_s2(thread_id)
            print(f"\n❌ Exception processing file {file_idx}: {e}")
            update_progress_s2(file_idx, input_file_path.name, 0, None, False, 0)

# Calculate final statistics
elapsed_time_overall = time.time() - start_time_overall

# Clear and show final summary
clear_output(wait=True)

print("\n" + "=" * 100)
print("📈 STAGE 2 PROCESSING COMPLETE")
print("=" * 100)

print(f"\n✅ Summary:")
print(f"   Files processed successfully: {len(successful_files):,}/{total_files:,}")
print(f"   Files failed: {len(failed_files):,}")
print(f"   Total sentences embedded: {total_sentences:,}")
print(f"   Embedding dimension: {STAGE_2_CONFIG['embedding_dim']}")
print(f"   Total processing time: {elapsed_time_overall/60:.2f} minutes")
if len(successful_files) > 0:
    print(f"   Average time per file: {elapsed_time_overall/len(successful_files):.2f} seconds")
    print(f"   Average sentences per file: {total_sentences/len(successful_files):.0f}")
print(f"   Processing speed: {len(successful_files)/(elapsed_time_overall/60):.1f} files/minute")

# Show sample of successful files
if successful_files:
    print(f"\n📋 Sample of Processed Files (first 10):")
    print("-" * 100)
    print(f"{'File #':<10} {'Sentences':<12} {'Output File':<30} {'Time (s)':<10}")
    print("-" * 100)
    sorted_files = sorted(successful_files, key=lambda x: x['file_number'])
    for file_info in sorted_files[:10]:
        print(f"{file_info['file_number']:<10} {file_info['sentences']:<12,} "
              f"{file_info['output']:<30} {file_info['time']:<10.2f}")
    
    if len(successful_files) > 10:
        print(f"\n   ... ({len(successful_files) - 10} more files)")

# Show failed files if any
if failed_files:
    print(f"\n⚠️  Failed Files ({len(failed_files)}):")
    for file_info in sorted(failed_files, key=lambda x: x['file_number']):
        print(f"   {file_info['file_number']}. {file_info['file_name']}")

# Save processing summary
summary_file = STAGE_2_OUTPUT_FOLDER / 'processing_summary.json'
summary_data = {
    'total_files': total_files,
    'successful_files': len(successful_files),
    'failed_files': len(failed_files),
    'total_sentences': total_sentences,
    'embedding_dimension': STAGE_2_CONFIG['embedding_dim'],
    'model_name': STAGE_2_CONFIG['model_name'],
    'processing_time_minutes': elapsed_time_overall / 60,
    'average_time_per_file_seconds': elapsed_time_overall / len(successful_files) if len(successful_files) > 0 else 0,
    'processing_speed_files_per_minute': len(successful_files) / (elapsed_time_overall / 60) if elapsed_time_overall > 0 else 0,
    'num_threads': num_threads,
    'batch_size': STAGE_2_CONFIG['batch_size'],
    'output_folder': str(STAGE_2_OUTPUT_FOLDER),
    'timestamp': datetime.now().isoformat()
}

with open(summary_file, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"\n💾 Processing summary saved: {summary_file.name}")

print("\n" + "=" * 100)
print("🎉 STAGE 2 COMPLETE!")
print("=" * 100)
print(f"\n💡 Next Stage: Topic Embedding Construction (Stage 3)")
print(f"   Input: {len(successful_files):,} files with {total_sentences:,} embedded sentences")
print(f"   Location: {STAGE_2_OUTPUT_FOLDER.absolute()}")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 2: VERIFICATION AND SAMPLE DISPLAY
# ============================================================================
# Verify output and display sample embeddings (both w3 and w5)
# ============================================================================

print("=" * 100)
print("STAGE 2: VERIFICATION - SAMPLE OUTPUT")
print("=" * 100)

# Get list of output files
output_files_s2 = sorted(STAGE_2_OUTPUT_FOLDER.glob('Data_s2_*.csv'))

if len(output_files_s2) > 0:
    print(f"\n✅ Found {len(output_files_s2):,} output files")
    
    # Load first file as sample
    sample_file_s2 = output_files_s2[0]
    print(f"\n📊 Loading sample file: {sample_file_s2.name}")
    print("   (This may take a moment due to large embedding columns...)")
    
    # Read only first few rows for quick verification
    sample_df_s2 = pd.read_csv(sample_file_s2, nrows=100)
    
    print(f"\n📊 Sample Data from: {sample_file_s2.name}")
    print(f"   Total sentences (in sample): {len(sample_df_s2):,}")
    print(f"   Columns: {list(sample_df_s2.columns)}")
    
    # Display first 3 sentences with context
    print(f"\n📋 First 3 Sentences (showing structure with both embeddings):")
    print("=" * 100)
    
    for idx in range(min(3, len(sample_df_s2))):
        row = sample_df_s2.iloc[idx]
        
        prev_1 = str(row['previous_sentence_1']) if pd.notna(row['previous_sentence_1']) else ""
        prev_2 = str(row['previous_sentence_2']) if pd.notna(row['previous_sentence_2']) else ""
        main_sent = str(row['main_sentence']) if pd.notna(row['main_sentence']) else ""
        next_1 = str(row['next_sentence_1']) if pd.notna(row['next_sentence_1']) else ""
        next_2 = str(row['next_sentence_2']) if pd.notna(row['next_sentence_2']) else ""
        
        # Parse embeddings (first few values)
        w3_embedding_str = str(row['w3_embedding'])
        w3_embedding_values = w3_embedding_str.split(',')[:5]  # Show first 5 values
        
        w5_embedding_str = str(row['w5_embedding'])
        w5_embedding_values = w5_embedding_str.split(',')[:5]  # Show first 5 values
        
        print(f"\n🔹 Sentence {idx + 1}:")
        print(f"   Sentence ID: {row['sentence_id']}")
        print(f"   Article ID: {row['article_id']}")
        print(f"   Date: {row['date']}")
        print(f"   Source: {row['source']}")
        print(f"   Context Window (5 sentences):")
        print(f"      Previous-1 (i-2): \"{prev_1[:40]}{'...' if len(prev_1) > 40 else ''}\"")
        print(f"      Previous-2 (i-1): \"{prev_2[:40]}{'...' if len(prev_2) > 40 else ''}\"")
        print(f"      Main (i):         \"{main_sent[:50]}{'...' if len(main_sent) > 50 else ''}\"")
        print(f"      Next-1 (i+1):     \"{next_1[:40]}{'...' if len(next_1) > 40 else ''}\"")
        print(f"      Next-2 (i+2):     \"{next_2[:40]}{'...' if len(next_2) > 40 else ''}\"")
        print(f"   w3_embedding (window 3): [{', '.join(w3_embedding_values[:5])}...] ({len(w3_embedding_values)} dims)")
        print(f"   w5_embedding (window 5): [{', '.join(w5_embedding_values[:5])}...] ({len(w5_embedding_values)} dims)")
    
    # Statistics
    print(f"\n\n📊 Statistics for Sample File:")
    print("-" * 100)
    
    # Count unique articles
    unique_articles = sample_df_s2['article_id'].nunique()
    avg_sentences_per_article = len(sample_df_s2) / unique_articles
    
    print(f"   Unique articles (in sample): {unique_articles:,}")
    print(f"   Average sentences per article: {avg_sentences_per_article:.1f}")
    
    # Verify embedding columns
    print(f"\n   Embedding Verification:")
    
    # Check w3 embedding
    sample_w3_embedding = sample_df_s2['w3_embedding'].iloc[0]
    w3_embedding_dim = len(str(sample_w3_embedding).split(','))
    print(f"      w3_embedding dimension: {w3_embedding_dim}")
    
    # Check w5 embedding
    sample_w5_embedding = sample_df_s2['w5_embedding'].iloc[0]
    w5_embedding_dim = len(str(sample_w5_embedding).split(','))
    print(f"      w5_embedding dimension: {w5_embedding_dim}")
    
    print(f"      Expected dimension: {STAGE_2_CONFIG['embedding_dim']}")
    
    if w3_embedding_dim == STAGE_2_CONFIG['embedding_dim'] and w5_embedding_dim == STAGE_2_CONFIG['embedding_dim']:
        print(f"      ✅ Both embedding dimensions match!")
    else:
        print(f"      ⚠️  Warning: Dimension mismatch!")
    
    # Count sentences with context
    has_prev_1 = sample_df_s2['previous_sentence_1'].notna() & (sample_df_s2['previous_sentence_1'] != "")
    has_prev_2 = sample_df_s2['previous_sentence_2'].notna() & (sample_df_s2['previous_sentence_2'] != "")
    has_next_1 = sample_df_s2['next_sentence_1'].notna() & (sample_df_s2['next_sentence_1'] != "")
    has_next_2 = sample_df_s2['next_sentence_2'].notna() & (sample_df_s2['next_sentence_2'] != "")
    
    has_full_w5 = has_prev_1 & has_prev_2 & has_next_1 & has_next_2
    has_full_w3 = has_prev_2 & has_next_1  # For w3, only need prev_2 (i-1) and next_1 (i+1)
    
    print(f"\n   Context Statistics:")
    print(f"      Sentences with full w5 context (2+1+2): {has_full_w5.sum():,} ({has_full_w5.sum()/len(sample_df_s2)*100:.1f}%)")
    print(f"      Sentences with full w3 context (1+1+1): {has_full_w3.sum():,} ({has_full_w3.sum()/len(sample_df_s2)*100:.1f}%)")
    
    # Display DataFrame info (excluding embedding columns for readability)
    print(f"\n\n📊 DataFrame Info (excluding embedding columns):")
    print("-" * 100)
    cols_to_show = [col for col in sample_df_s2.columns if col not in ['w3_embedding', 'w5_embedding']]
    sample_df_s2[cols_to_show].info()
    
    # Display sample without embedding columns
    print(f"\n\n📋 Sample Data (first 10 rows, excluding embedding columns):")
    print("=" * 100)
    display(sample_df_s2[cols_to_show].head(10))
    
    # Test: Parse both embeddings back to arrays
    print(f"\n\n🧪 Embedding Test:")
    print("-" * 100)
    
    # Test w3 embedding
    test_w3_str = sample_df_s2['w3_embedding'].iloc[0]
    test_w3_array = np.array([float(x) for x in test_w3_str.split(',')])
    print(f"   w3_embedding (Window 3):")
    print(f"      Successfully parsed to numpy array")
    print(f"      Shape: {test_w3_array.shape}")
    print(f"      First 10 values: {test_w3_array[:10]}")
    print(f"      Min: {test_w3_array.min():.4f}, Max: {test_w3_array.max():.4f}")
    print(f"      Mean: {test_w3_array.mean():.4f}, Std: {test_w3_array.std():.4f}")
    
    # Test w5 embedding
    test_w5_str = sample_df_s2['w5_embedding'].iloc[0]
    test_w5_array = np.array([float(x) for x in test_w5_str.split(',')])
    print(f"\n   w5_embedding (Window 5):")
    print(f"      Successfully parsed to numpy array")
    print(f"      Shape: {test_w5_array.shape}")
    print(f"      First 10 values: {test_w5_array[:10]}")
    print(f"      Min: {test_w5_array.min():.4f}, Max: {test_w5_array.max():.4f}")
    print(f"      Mean: {test_w5_array.mean():.4f}, Std: {test_w5_array.std():.4f}")
    
    # Compare embeddings
    print(f"\n   Comparison:")
    cosine_sim = np.dot(test_w3_array, test_w5_array) / (np.linalg.norm(test_w3_array) * np.linalg.norm(test_w5_array))
    print(f"      Cosine similarity between w3 and w5: {cosine_sim:.4f}")
    print(f"      (Higher values indicate similar embeddings despite different window sizes)")
    
else:
    print("\n⚠️  No output files found. Please run the processing cell first.")

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE")
print("=" * 100)

# 📊 STAGE 3: Soft Labeling - Topic Embeddings

---

## Overview

Generate **ONE embedding per TOPIC** for soft labeling.

### Process:
1. Read ALL articles from `Soft_Labeling_Topic_Articles/`
2. Generate SBERT embeddings for each article
3. Mean pooling ALL articles per topic (across all subtopics)
4. Save to `Processed_Data/topic_embeddings.json`

### Output Format:
```json
{
  "War": [0.123, -0.456, ...],        // 768-dim
  "Health": [0.234, -0.567, ...],
  "Technology": [...],
  "Climate": [...],
  "Economics": [...]
}
```

---

In [2]:
# ============================================================================
# STAGE 3: IMPORTS - Required Libraries for Soft Labeling
# ============================================================================

print("=" * 100)
print("🔧 IMPORTING LIBRARIES FOR STAGE 3: SOFT LABELING")
print("=" * 100)

# Standard libraries
import os
import json
import glob
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict

# Data manipulation
import numpy as np
import pandas as pd

# SBERT for embeddings
from sentence_transformers import SentenceTransformer
import torch

# Progress tracking
from tqdm.auto import tqdm

print("\n✅ All libraries imported successfully!")
print(f"📦 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print("=" * 100)

🔧 IMPORTING LIBRARIES FOR STAGE 3: SOFT LABELING

✅ All libraries imported successfully!
📦 PyTorch version: 2.10.0+cu128
🎮 CUDA available: True
   GPU: NVIDIA GeForce MX450
   VRAM: 1.64 GB


In [3]:
# ============================================================================
# STAGE 3: INSTALL ADDITIONAL DEPENDENCIES (if needed)
# ============================================================================

# Install scipy for cosine similarity calculation
import subprocess
import sys

print("=" * 100)
print("📦 INSTALLING ADDITIONAL DEPENDENCIES")
print("=" * 100)

try:
    import scipy
    print("✅ scipy already installed")
except ImportError:
    print("⏳ Installing scipy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scipy"])
    print("✅ scipy installed successfully!")

print("=" * 100)

📦 INSTALLING ADDITIONAL DEPENDENCIES
✅ scipy already installed


In [4]:
# ============================================================================
# STAGE 3: CONFIGURATION - Paths and Settings
# ============================================================================

print("\n" + "=" * 100)
print("⚙️  CONFIGURATION FOR STAGE 3")
print("=" * 100)

# Base paths
BASE_DIR = Path("/home/hp/SEM2/INLP/Naretve_Shift")
SOFT_LABEL_DIR = BASE_DIR / "Soft_Labeling_Topic_Articles"
OUTPUT_DIR = BASE_DIR / "Processed_Data"

# Output file
OUTPUT_FILE = OUTPUT_DIR / "topic_embeddings.json"

# SBERT configuration
SBERT_MODEL_NAME = "all-mpnet-base-v2"  # 768-dim embeddings
BATCH_SIZE = 32  # Adjust based on GPU memory
USE_GPU = torch.cuda.is_available()

# Topic structure (as per your directory)
TOPICS = {
    "War": [
        "Armed_Conflict",
        "Geopolitics", 
        "Humanitarian_Crisis",
        "Defense_Technology",
        "Peace_Process"
    ],
    "Health": [
        "Public_Health",
        "Healthcare_Policy",
        "Medical_Research",
        "Mental_Health",
        "Chronic_Disease"
    ],
    "Technology": [
        "AI_Automation",
        "Digital_Infrastructure",
        "Consumer_Tech",
        "Tech_Regulation",
        "Tech_Labor_Impact"
    ],
    "Climate": [
        "Global_Warming",
        "Renewable_Energy",
        "Environmental_Disasters",
        "Climate_Policy",
        "Sustainability"
    ],
    "Economics": [
        "Macroeconomics",
        "Global_Trade",
        "Financial_Markets",
        "Fiscal_Policy",
        "Corporate_Economy"
    ]
}

print(f"\n📂 Input Directory: {SOFT_LABEL_DIR}")
print(f"📁 Output Directory: {OUTPUT_DIR}")
print(f"\n📊 Topics Configuration:")
for topic, subtopics in TOPICS.items():
    print(f"   {topic}: {len(subtopics)} subtopics")
print(f"\n🤖 SBERT Model: {SBERT_MODEL_NAME}")
print(f"📦 Batch Size: {BATCH_SIZE}")
print(f"🎮 Using GPU: {USE_GPU}")
print("=" * 100)


⚙️  CONFIGURATION FOR STAGE 3

📂 Input Directory: /home/hp/SEM2/INLP/Naretve_Shift/Soft_Labeling_Topic_Articles
📁 Output Directory: /home/hp/SEM2/INLP/Naretve_Shift/Processed_Data

📊 Topics Configuration:
   War: 5 subtopics
   Health: 5 subtopics
   Technology: 5 subtopics
   Climate: 5 subtopics
   Economics: 5 subtopics

🤖 SBERT Model: all-mpnet-base-v2
📦 Batch Size: 32
🎮 Using GPU: True


In [ ]:
# ============================================================================
# STAGE 3: LOAD SBERT MODEL
# ============================================================================

print("\n" + "=" * 100)
print("🤖 LOADING SBERT MODEL")
print("=" * 100)

# Load SBERT model
device = "cuda" if USE_GPU else "cpu"
print(f"\n📥 Loading {SBERT_MODEL_NAME} on {device.upper()}...")

model = SentenceTransformer(SBERT_MODEL_NAME, device=device)

# Enable FP16 for GPU (faster inference)
if USE_GPU:
    model = model.half()
    print("✅ Enabled FP16 mixed precision for faster GPU inference")

print(f"✅ Model loaded successfully!")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"   Max sequence length: {model.max_seq_length}")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 3: HELPER FUNCTIONS
# ============================================================================

def read_article(file_path: Path) -> str:
    """
    Read article text from file.
    
    Args:
        file_path: Path to article text file
        
    Returns:
        Article text content
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read().strip()
    except Exception as e:
        print(f"⚠️  Error reading {file_path}: {e}")
        return ""


def get_all_articles_for_topic(topic: str) -> List[Tuple[str, str, str]]:
    """
    Get ALL article files for a given topic (across all subtopics).
    
    Args:
        topic: Main topic (e.g., "War", "Health")
        
    Returns:
        List of tuples (subtopic, filename, article_text)
    """
    topic_dir = SOFT_LABEL_DIR / topic
    
    if not topic_dir.exists():
        print(f"⚠️  Directory not found: {topic_dir}")
        return []
    
    articles = []
    
    # Iterate through all subtopic directories
    for subtopic_dir in topic_dir.iterdir():
        if subtopic_dir.is_dir():
            subtopic = subtopic_dir.name
            
            # Find all .txt files in this subtopic
            article_files = list(subtopic_dir.glob("*.txt"))
            
            for file_path in article_files:
                text = read_article(file_path)
                if text:  # Only include non-empty articles
                    articles.append((subtopic, file_path.name, text))
    
    return articles


def generate_topic_embedding(articles: List[Tuple[str, str, str]], 
                             batch_size: int = 32) -> np.ndarray:
    """
    Generate mean pooled embedding for entire topic (all articles across all subtopics).
    
    Args:
        articles: List of (subtopic, filename, text) tuples
        batch_size: Batch size for encoding
        
    Returns:
        Mean pooled embedding vector (768-dim)
    """
    if not articles:
        return None
    
    # Extract texts only
    texts = [text for _, _, text in articles]
    
    # Generate embeddings for all articles
    print(f"   🔄 Encoding {len(texts)} articles...")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True  # L2 normalization
    )
    
    # Mean pooling across ALL articles
    mean_embedding = np.mean(embeddings, axis=0)
    
    # Re-normalize after mean pooling
    mean_embedding = mean_embedding / np.linalg.norm(mean_embedding)
    
    return mean_embedding


print("✅ Helper functions defined successfully!")
print("   - read_article()")
print("   - get_all_articles_for_topic()")
print("   - generate_topic_embedding()")

In [ ]:
# ============================================================================
# STAGE 3: MAIN PROCESSING - Generate Topic Embeddings
# ============================================================================

print("\n" + "=" * 100)
print("🚀 STARTING STAGE 3: TOPIC EMBEDDING GENERATION")
print("=" * 100)

# Dictionary to store all topic embeddings
topic_embeddings = {}

# Statistics tracking
stats = {
    "total_topics": 0,
    "total_articles": 0,
    "articles_by_topic": {},
    "subtopics_by_topic": {}
}

# Process each topic
for topic in TOPICS.keys():
    print(f"\n{'=' * 100}")
    print(f"📌 PROCESSING TOPIC: {topic}")
    print(f"{'=' * 100}")
    
    stats["total_topics"] += 1
    
    # Get ALL articles for this topic (across all subtopics)
    articles = get_all_articles_for_topic(topic)
    
    if not articles:
        print(f"⚠️  No articles found for topic: {topic}")
        continue
    
    # Count subtopics
    subtopics = set(subtopic for subtopic, _, _ in articles)
    stats["subtopics_by_topic"][topic] = len(subtopics)
    stats["articles_by_topic"][topic] = len(articles)
    stats["total_articles"] += len(articles)
    
    print(f"\n📊 Found {len(articles)} articles across {len(subtopics)} subtopics:")
    
    # Show breakdown by subtopic
    subtopic_counts = {}
    for subtopic, _, _ in articles:
        subtopic_counts[subtopic] = subtopic_counts.get(subtopic, 0) + 1
    
    for subtopic, count in sorted(subtopic_counts.items()):
        print(f"   └─ {subtopic}: {count} articles")
    
    # Generate embedding for entire topic
    try:
        print(f"\n🔄 Generating embedding for {topic}...")
        
        embedding = generate_topic_embedding(articles, batch_size=BATCH_SIZE)
        
        if embedding is not None:
            # Store embedding
            topic_embeddings[topic] = embedding.tolist()
            
            print(f"   ✅ {topic} embedding generated successfully!")
            print(f"      Embedding shape: {embedding.shape}")
            print(f"      L2 norm: {np.linalg.norm(embedding):.6f}")
        
    except Exception as e:
        print(f"   ❌ Error processing {topic}: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    # Clear GPU cache after each topic
    if USE_GPU:
        torch.cuda.empty_cache()
        print(f"   🧹 GPU cache cleared")

print("\n" + "=" * 100)
print("💾 SAVING TOPIC EMBEDDINGS TO JSON")
print("=" * 100)

# Save to single JSON file
output_file = OUTPUT_DIR / "topic_embeddings.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(topic_embeddings, f, indent=2, ensure_ascii=False)

print(f"\n✅ Topic embeddings saved to: {output_file.relative_to(BASE_DIR)}")
print(f"   File size: {output_file.stat().st_size / 1024:.2f} KB")

print("\n" + "=" * 100)
print("🎉 STAGE 3 PROCESSING COMPLETE!")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 3: STATISTICS & SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("STAGE 3: PROCESSING STATISTICS")
print("=" * 100)

print(f"\nOverall Statistics:")
print(f"   Total Topics:          {stats['total_topics']}")
print(f"   Total Articles:        {stats['total_articles']}")
print(f"   Embeddings Generated:  {len(topic_embeddings)}")

print(f"\nTopic-wise Breakdown:")
print(f"{'Topic':<15} {'Subtopics':<12} {'Articles':<10} {'Status'}")
print("=" * 60)

for topic in TOPICS.keys():
    subtopics = stats['subtopics_by_topic'].get(topic, 0)
    articles = stats['articles_by_topic'].get(topic, 0)
    status = "[DONE]" if topic in topic_embeddings else "[FAILED]"
    print(f"{topic:<15} {subtopics:<12} {articles:<10} {status}")

print("\n" + "=" * 60)
print(f"{'TOTAL':<15} {sum(stats['subtopics_by_topic'].values()):<12} {stats['total_articles']:<10}")

print(f"\n[OUTPUT] {OUTPUT_DIR.relative_to(BASE_DIR)}/topic_embeddings.json")

print("\n" + "=" * 100)
print("STAGE 3 COMPLETE - All topic embeddings saved!")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 3: LOAD & VERIFY EMBEDDINGS
# ============================================================================

print("\n" + "=" * 100)
print("VERIFICATION: Loading and Checking Topic Embeddings")
print("=" * 100)

# Load the generated topic_embeddings.json
embeddings_file = OUTPUT_DIR / "topic_embeddings.json"

if embeddings_file.exists():
    with open(embeddings_file, 'r') as f:
        loaded_embeddings = json.load(f)
    
    print(f"\n[OK] Embeddings file loaded successfully!")
    print(f"   File: {embeddings_file.relative_to(BASE_DIR)}")
    print(f"   Topics found: {len(loaded_embeddings)}")
    
    print(f"\nEmbedding Details per Topic:")
    print(f"{'Topic':<15} {'Embedding Dim':<15} {'L2 Norm':<12} {'Mean':<12} {'Std'}")
    print("=" * 80)
    
    for topic, embedding in loaded_embeddings.items():
        emb_array = np.array(embedding)
        l2_norm = np.linalg.norm(emb_array)
        mean = emb_array.mean()
        std = emb_array.std()
        
        print(f"{topic:<15} {len(embedding):<15} {l2_norm:<12.6f} {mean:<12.6f} {std:.6f}")
    
    print("\n" + "=" * 80)
    
    # Show sample of one embedding
    sample_topic = list(loaded_embeddings.keys())[0]
    sample_embedding = np.array(loaded_embeddings[sample_topic])
    
    print(f"\nSample Embedding ({sample_topic}):")
    print(f"   First 10 values: {sample_embedding[:10]}")
    print(f"   Shape: {sample_embedding.shape}")
    print(f"   L2 Norm: {np.linalg.norm(sample_embedding):.6f} (should be ~1.0)")
    
else:
    print(f"[WARNING] Embeddings file not found: {embeddings_file}")

print("\n" + "=" * 100)
print("VERIFICATION COMPLETE")
print("=" * 100)

In [ ]:
# ============================================================================
# STAGE 3: SOFT LABELING EXAMPLE - Label New Article
# ============================================================================

print("\n" + "=" * 100)
print("EXAMPLE: Soft Labeling a New Article")
print("=" * 100)

# Example new article (you can replace this with your own)
new_article = """
The ongoing conflict in the region has escalated significantly, with armed forces 
engaging in intense battles near the border. Military experts warn that the situation 
could deteriorate further without immediate diplomatic intervention.
"""

print(f"\nNew Article to Label:")
print(f"{new_article.strip()}")

# Generate embedding for new article
print(f"\nGenerating embedding for new article...")
new_article_embedding = model.encode(
    new_article, 
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Load topic embeddings
embeddings_file = OUTPUT_DIR / "topic_embeddings.json"

if embeddings_file.exists():
    with open(embeddings_file, 'r') as f:
        topic_embeddings_data = json.load(f)
    
    print(f"[OK] Loaded {len(topic_embeddings_data)} topic embeddings")
    
    # Calculate cosine similarity with all topics
    from scipy.spatial.distance import cosine
    
    similarities = {}
    for topic, embedding in topic_embeddings_data.items():
        topic_emb = np.array(embedding)
        similarity = 1 - cosine(new_article_embedding, topic_emb)
        similarities[topic] = similarity
    
    # Sort by similarity
    sorted_topics = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    
    print(f"\nTopic Similarity Scores:")
    print(f"{'Rank':<6} {'Topic':<15} {'Similarity':<12} {'Confidence'}")
    print("=" * 60)
    
    for rank, (topic, similarity) in enumerate(sorted_topics, 1):
        confidence = "[HIGH]" if similarity > 0.7 else "[MED]" if similarity > 0.5 else "[LOW]"
        print(f"{rank:<6} {topic:<15} {similarity:>8.4f}      {confidence}")
    
    # Best match
    best_topic, best_score = sorted_topics[0]
    print(f"\n[BEST MATCH] {best_topic} (similarity: {best_score:.4f})")
    
    # Soft label with confidence distribution
    print(f"\nSoft Label Distribution (Normalized):")
    total_sim = sum(sim for _, sim in similarities.items())
    
    for topic, sim in sorted(similarities.items(), key=lambda x: x[1], reverse=True):
        weight = sim / total_sim
        bar = "#" * int(weight * 50)
        print(f"   {topic:<15} {weight*100:>5.2f}% {bar}")
    
else:
    print(f"[ERROR] Embeddings file not found: {embeddings_file}")

print("\n" + "=" * 100)
print("SOFT LABELING EXAMPLE COMPLETE")
print("=" * 100)

In [1]:
# ============================================================================
# STAGE 3: CLEANUP & FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("CLEANUP & FINAL SUMMARY")
print("=" * 100)

# Clear GPU cache
if USE_GPU:
    torch.cuda.empty_cache()
    print("\n[OK] GPU cache cleared")

print(f"\nFinal Summary:")
print(f"   Output: {OUTPUT_DIR.relative_to(BASE_DIR)}/topic_embeddings.json")
print(f"   Topics: {len(topic_embeddings)}")
print(f"   Articles: {stats['total_articles']}")

print("\n" + "=" * 100)
print("STAGE 3 COMPLETE!")
print("=" * 100)


CLEANUP & FINAL SUMMARY


NameError: name 'USE_GPU' is not defined

# STAGE 3.2: Data Soft Labeling

---

## Overview

Apply soft labels to Stage 2 data by calculating similarity between sentence embeddings and topic embeddings.

### Process:
1. Load topic embeddings from `topic_embeddings.json`
2. Read Stage 2 CSV files (with `w3_embedding`)
3. Calculate cosine similarity for each sentence
4. Add 5 columns: `War`, `Health`, `Technology`, `Climate`, `Economics`
5. Save to `Processed_Data/Stage_3/`

### Input:
- `Processed_Data/Stage_2/*.csv` (with w3_embedding column)
- `Processed_Data/topic_embeddings.json`

### Output:
- `Processed_Data/Stage_3/*.csv` (with 5 topic similarity columns)

---

In [2]:
# ============================================================================
# STAGE 3.2: CONFIGURATION
# ============================================================================

print("\n" + "=" * 100)
print("STAGE 3.2: DATA SOFT LABELING - CONFIGURATION")
print("=" * 100)

# Paths
STAGE2_DIR = BASE_DIR / "Processed_Data" / "Stage_2"
STAGE3_OUTPUT_DIR = BASE_DIR / "Processed_Data" / "Stage_3_w5"
TOPIC_EMBEDDINGS_FILE = BASE_DIR / "Processed_Data" / "topic_embeddings.json"

# Create output directory
STAGE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nInput Directory:  {STAGE2_DIR.relative_to(BASE_DIR)}")
print(f"Output Directory: {STAGE3_OUTPUT_DIR.relative_to(BASE_DIR)}")
print(f"Topic Embeddings: {TOPIC_EMBEDDINGS_FILE.relative_to(BASE_DIR)}")

# Check if directories exist
if not STAGE2_DIR.exists():
    print(f"\n[ERROR] Stage 2 directory not found: {STAGE2_DIR}")
else:
    csv_files = list(STAGE2_DIR.glob("*.csv"))
    print(f"\n[INFO] Found {len(csv_files)} CSV files in Stage 2")

if not TOPIC_EMBEDDINGS_FILE.exists():
    print(f"\n[ERROR] Topic embeddings not found: {TOPIC_EMBEDDINGS_FILE}")
    print("[INFO] Please run Stage 3 (topic embedding generation) first!")
else:
    print(f"[OK] Topic embeddings file found")

print("=" * 100)


STAGE 3.2: DATA SOFT LABELING - CONFIGURATION


NameError: name 'BASE_DIR' is not defined

In [3]:
# ============================================================================
# STAGE 3.2: LOAD TOPIC EMBEDDINGS
# ============================================================================

print("\n" + "=" * 100)
print("LOADING TOPIC EMBEDDINGS")
print("=" * 100)

# Load topic embeddings
with open(TOPIC_EMBEDDINGS_FILE, 'r') as f:
    topic_embeddings_dict = json.load(f)

# Convert to numpy arrays
topic_embeddings_np = {
    topic: np.array(embedding) 
    for topic, embedding in topic_embeddings_dict.items()
}

print(f"\n[OK] Loaded {len(topic_embeddings_np)} topic embeddings:")
for topic, embedding in topic_embeddings_np.items():
    print(f"   {topic}: {embedding.shape}")

print("=" * 100)


LOADING TOPIC EMBEDDINGS


NameError: name 'TOPIC_EMBEDDINGS_FILE' is not defined

In [4]:
# ============================================================================
# STAGE 3.2: HELPER FUNCTIONS (GPU-OPTIMIZED)
# ============================================================================

def parse_embedding_string(embedding_str):
    """
    Parse embedding string from CSV to numpy array.
    Handles formats like: '[0.123, -0.456, ...]' or '0.123 -0.456 ...'
    """
    if pd.isna(embedding_str):
        return None
    
    # Convert string to array
    embedding_str = str(embedding_str).strip()
    
    # Remove brackets if present
    embedding_str = embedding_str.replace('[', '').replace(']', '')
    
    # Split by comma or space
    if ',' in embedding_str:
        values = [float(x.strip()) for x in embedding_str.split(',') if x.strip()]
    else:
        values = [float(x.strip()) for x in embedding_str.split() if x.strip()]
    
    return np.array(values)


def batch_calculate_similarities_gpu(sentence_embeddings, topic_embeddings_dict, batch_size=1024):
    """
    GPU-accelerated batch similarity calculation.
    
    Args:
        sentence_embeddings: numpy array of shape (N, 768)
        topic_embeddings_dict: Dict of topic embeddings
        batch_size: Batch size for GPU processing
    
    Returns:
        dict: {topic_name: similarity_scores_array}
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Convert to torch tensor
    sent_embeddings_tensor = torch.from_numpy(sentence_embeddings).float().to(device)
    
    # Normalize sentence embeddings (for cosine similarity)
    sent_embeddings_normalized = torch.nn.functional.normalize(sent_embeddings_tensor, p=2, dim=1)
    
    # Calculate similarities for each topic
    similarities_dict = {}
    
    for topic_name, topic_embedding in topic_embeddings_dict.items():
        # Convert topic embedding to tensor
        topic_tensor = torch.from_numpy(topic_embedding).float().to(device)
        topic_normalized = torch.nn.functional.normalize(topic_tensor.unsqueeze(0), p=2, dim=1)
        
        # Calculate cosine similarity (batch matrix multiplication)
        # Shape: (N, 1) -> (N,)
        similarities = torch.mm(sent_embeddings_normalized, topic_normalized.t()).squeeze()
        
        # Move back to CPU and convert to numpy
        similarities_dict[topic_name] = similarities.cpu().numpy()
    
    # Clear GPU cache
    if device == "cuda":
        torch.cuda.empty_cache()
    
    return similarities_dict


def add_soft_labels_to_dataframe_gpu(df, topic_embeddings_dict, embedding_column='w3_embedding', suffix='', verbose=False):
    """
    GPU-accelerated soft label addition to dataframe.
    Optimized for multi-threading with minimal prints.
    
    Args:
        df: Input dataframe with embedding column
        topic_embeddings_dict: Dict of topic embeddings
        embedding_column: Name of embedding column to use ('w3_embedding' or 'w5_embedding')
        suffix: Suffix to add to column names (e.g., '_w5' for w5 embeddings)
        verbose: Print detailed progress (default: False for threading)
    
    Returns:
        DataFrame with added topic similarity columns
    """
    # Step 1: Parse all embeddings (silent)
    embeddings_list = []
    valid_indices = []
    
    for idx, row in df.iterrows():
        embedding = parse_embedding_string(row[embedding_column])
        if embedding is not None:
            embeddings_list.append(embedding)
            valid_indices.append(idx)
    
    if len(embeddings_list) == 0:
        return df
    
    # Step 2: Convert to numpy array
    embeddings_array = np.array(embeddings_list)
    
    # Step 3: Calculate similarities on GPU
    similarities_dict = batch_calculate_similarities_gpu(
        embeddings_array, 
        topic_embeddings_dict
    )
    
    # Step 4: Add to dataframe
    # Initialize columns with 0.0 (with suffix if provided)
    for topic in topic_embeddings_dict.keys():
        column_name = f"{topic}{suffix}"
        df[column_name] = 0.0
    
    # Fill in calculated values
    for topic_name, similarities in similarities_dict.items():
        column_name = f"{topic_name}{suffix}"
        for i, idx in enumerate(valid_indices):
            df.at[idx, column_name] = float(similarities[i])
    
    return df


# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n[OK] Helper functions defined (GPU-optimized):")
print(f"   Device: {device}")
print(f"   - parse_embedding_string()")
print(f"   - batch_calculate_similarities_gpu()")
print(f"   - add_soft_labels_to_dataframe_gpu()")

if device == "cuda":
    print(f"\n   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

NameError: name 'torch' is not defined

In [5]:
# ============================================================================
# STAGE 3.2: MAIN PROCESSING - Multi-threaded (Optimized for 8GB RAM)
# ============================================================================

import threading
from queue import Queue
import time
from collections import defaultdict
from IPython.display import clear_output
import re

print("\n" + "=" * 100)
print("STAGE 3.2: MULTI-THREADED SOFT LABELING")
print("=" * 100)

# Get all CSV files from Stage 2
csv_files = sorted(STAGE2_DIR.glob("*.csv"))

# Filter out already processed files
files_to_process = []
already_processed = []

for csv_file in csv_files:
    # Extract file number and create expected output filename
    # "Data_s2_107.csv" -> "Data_s3_107.csv"
    match = re.search(r'Data_s2_(\d+)\.csv', csv_file.name, re.IGNORECASE)
    if match:
        file_num = match.group(1)
        output_filename = f"Data_s3_{file_num}.csv"
    else:
        output_filename = csv_file.name
    
    output_file = STAGE3_OUTPUT_DIR / output_filename
    
    # Check if output file already exists
    if output_file.exists():
        already_processed.append(csv_file.name)
    else:
        files_to_process.append(csv_file)

print(f"\n[INFO] Total files in Stage 2: {len(csv_files)}")
print(f"[INFO] Already processed: {len(already_processed)}")
print(f"[INFO] Files to process: {len(files_to_process)}")

if already_processed:
    print(f"\n[SKIP] Already processed files (first 5):")
    for fname in already_processed[:5]:
        print(f"   ✓ {fname}")
    if len(already_processed) > 5:
        print(f"   ... and {len(already_processed) - 5} more")

# Update csv_files to only include files that need processing
csv_files = files_to_process

if len(csv_files) == 0:
    print(f"\n✅ All files already processed! Nothing to do.")
else:
    print(f"\n[PROCESS] Will process {len(csv_files)} files:")
    for fname in [f.name for f in csv_files[:5]]:
        print(f"   → {fname}")
    if len(csv_files) > 5:
        print(f"   ... and {len(csv_files) - 5} more")

# Configuration for 8GB RAM with ~700MB files
MAX_WORKERS = 6 # Adjust based on your system

print(f"\n[CONFIG] Worker threads: {MAX_WORKERS}")
print(f"[CONFIG] Estimated RAM usage: ~{MAX_WORKERS * 0.7:.1f} GB")

# Thread-safe statistics with progress tracking
stats_lock = threading.Lock()
stats_s32 = {
    'total_files': len(csv_files),
    'processed_files': 0,
    'failed_files': 0,
    'total_rows': 0,
    'processing': {},  # {thread_id: (filename, progress%)}
    'completed': []  # List of (filename, status, time)
}

start_time = time.time()


def process_single_file(csv_file, thread_id):
    """Process a single file in a thread with progress tracking"""
    
    file_start = time.time()
    
    try:
        # Update status
        with stats_lock:
            stats_s32['processing'][thread_id] = (csv_file.name, 0)
        
        # Load CSV (10% progress)
        df = pd.read_csv(csv_file)
        with stats_lock:
            stats_s32['processing'][thread_id] = (csv_file.name, 10)
        
        # Check if required embedding column exists
        if 'w3_embedding' not in df.columns:
            with stats_lock:
                stats_s32['failed_files'] += 1
                if thread_id in stats_s32['processing']:
                    del stats_s32['processing'][thread_id]
                file_time = time.time() - file_start
                stats_s32['completed'].append((csv_file.name, 'FAIL', file_time))
            return False
        
        # Process w5_embedding only (20% -> 80% progress)
        with stats_lock:
            stats_s32['processing'][thread_id] = (csv_file.name, 20)
        
        df = add_soft_labels_to_dataframe_gpu(
            df, 
            topic_embeddings_np, 
            embedding_column='w5_embedding',
            suffix='',  # No suffix - just topic names
            verbose=False
        )
        
        with stats_lock:
            stats_s32['processing'][thread_id] = (csv_file.name, 80)
        
        # Extract file number and create output filename
        # "Data_s2_107.csv" -> "Data_s3_107.csv" (lowercase s)
        match = re.search(r'Data_s2_(\d+)\.csv', csv_file.name, re.IGNORECASE)
        if match:
            file_num = match.group(1)
            output_filename = f"Data_s3_{file_num}.csv"
        else:
            # Fallback: keep original name
            output_filename = csv_file.name
        
        # Save to Stage 3 (80% -> 100% progress)
        with stats_lock:
            stats_s32['processing'][thread_id] = (csv_file.name, 90)
        
        output_file = STAGE3_OUTPUT_DIR / output_filename
        df.to_csv(output_file, index=False)
        
        # Update statistics
        with stats_lock:
            stats_s32['processing'][thread_id] = (csv_file.name, 100)
            stats_s32['processed_files'] += 1
            stats_s32['total_rows'] += len(df)
            file_time = time.time() - file_start
            stats_s32['completed'].append((output_filename, 'OK', file_time))
            if thread_id in stats_s32['processing']:
                del stats_s32['processing'][thread_id]
        
        return True
        
    except Exception as e:
        with stats_lock:
            stats_s32['failed_files'] += 1
            file_time = time.time() - file_start
            stats_s32['completed'].append((csv_file.name, f'ERROR: {str(e)[:30]}', file_time))
            if thread_id in stats_s32['processing']:
                del stats_s32['processing'][thread_id]
        return False


def worker(file_queue, thread_id):
    """Worker thread that processes files from queue"""
    
    while True:
        try:
            csv_file = file_queue.get(timeout=1)
            if csv_file is None:  # Poison pill
                break
            
            # Process file
            success = process_single_file(csv_file, thread_id)
            
            file_queue.task_done()
            
        except:
            break


def display_progress():
    """Display simple progress summary"""
    with stats_lock:
        completed = stats_s32['processed_files'] + stats_s32['failed_files']
        percent = (completed / stats_s32['total_files'] * 100) if stats_s32['total_files'] > 0 else 0
        elapsed = time.time() - start_time
        
        print(f"\n{'=' * 80}")
        print(f"PROGRESS: {completed}/{stats_s32['total_files']} files ({percent:.1f}%)")
        print(f"Elapsed: {elapsed:.0f}s | OK: {stats_s32['processed_files']} | Failed: {stats_s32['failed_files']}")
        print(f"{'=' * 80}")
        
        # Show active threads
        if stats_s32['processing']:
            print(f"\nActive Threads:")
            for tid, (fname, progress) in stats_s32['processing'].items():
                bar_len = int(progress / 5)
                bar = '█' * bar_len + '░' * (20 - bar_len)
                print(f"  T{tid:2d}: [{bar}] {progress:3d}% | {fname}")
        
        # Show last 5 completed
        if stats_s32['completed']:
            print(f"\nRecently Completed (last 5):")
            for fname, status, ftime in stats_s32['completed'][-5:]:
                print(f"  [{status:6s}] {fname} ({ftime:.1f}s)")
        
        print(f"{'=' * 80}\n")


# Only proceed if there are files to process
if len(csv_files) == 0:
    print(f"\n{'=' * 100}")
    print(f"✅ ALL FILES ALREADY PROCESSED - NOTHING TO DO")
    print(f"{'=' * 100}")
else:
    # Create queue and add files
    file_queue = Queue()
    for csv_file in csv_files:
        file_queue.put(csv_file)

    # Start workers
    print(f"\n[INFO] Starting {MAX_WORKERS} worker threads...\n")
    threads = []

    for i in range(MAX_WORKERS):
        t = threading.Thread(target=worker, args=(file_queue, i+1))
        t.daemon = True
        t.start()
        threads.append(t)

    # Monitor progress
    try:
        last_display = 0
        
        while True:
            # Display progress every 10 seconds
            current_time = time.time()
            if current_time - last_display >= 10:
                clear_output(wait=True)
                display_progress()
                last_display = current_time
            
            # Check if all done
            if file_queue.empty() and all(not t.is_alive() for t in threads):
                break
            
            time.sleep(1)
        
    except KeyboardInterrupt:
        print("\n\n[WARN] Interrupted by user")

        # Wait for completion
        file_queue.join()

        # Send poison pills
        for _ in range(MAX_WORKERS):
            file_queue.put(None)

        # Wait for threads to finish
        for t in threads:
            t.join()

    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elapsed_time = time.time() - start_time

    # Final display
    clear_output(wait=True)
    print(f"\n{'=' * 100}")
    print(f"STAGE 3.2 PROCESSING COMPLETE")
    print(f"Time elapsed: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    if len(csv_files) > 0:
        print(f"Speed: {len(csv_files) / (elapsed_time/60):.2f} files/minute")
    print(f"{'=' * 100}")


STAGE 3.2: MULTI-THREADED SOFT LABELING


NameError: name 'STAGE2_DIR' is not defined

In [8]:
# ============================================================================
# STAGE 3.2: STATISTICS & SUMMARY
# ============================================================================

print("\n" + "=" * 100)
print("STAGE 3.2: PROCESSING STATISTICS")
print("=" * 100)

print(f"\nOverall Statistics:")
print(f"   Total Files:       {stats_s32['total_files']}")
print(f"   Processed Files:   {stats_s32['processed_files']}")
print(f"   Failed Files:      {stats_s32['failed_files']}")
print(f"   Total Rows:        {stats_s32['total_rows']:,}")

print(f"\nOutput Location: {STAGE3_OUTPUT_DIR.relative_to(BASE_DIR)}")

# List output files
output_files = sorted(STAGE3_OUTPUT_DIR.glob("*.csv"))
print(f"\nOutput Files ({len(output_files)}):")
for i, file in enumerate(output_files[:10], 1):
    file_size = file.stat().st_size / (1024 * 1024)
    print(f"   {i}. {file.name} ({file_size:.2f} MB)")

if len(output_files) > 10:
    print(f"   ... and {len(output_files) - 10} more files")

print("\n" + "=" * 100)
print("STAGE 3.2 COMPLETE - Soft labels added to all files!")
print("=" * 100)


STAGE 3.2: PROCESSING STATISTICS

Overall Statistics:
   Total Files:       65
   Processed Files:   65
   Failed Files:      0
   Total Rows:        1,760,033

Output Location: Processed_Data/Stage_3

Output Files (113):
   1. Data_s3_1.csv (282.93 MB)
   2. Data_s3_10.csv (684.19 MB)
   3. Data_s3_100.csv (811.94 MB)
   4. Data_s3_101.csv (746.87 MB)
   5. Data_s3_102.csv (795.00 MB)
   6. Data_s3_103.csv (672.92 MB)
   7. Data_s3_104.csv (791.02 MB)
   8. Data_s3_105.csv (722.55 MB)
   9. Data_s3_106.csv (709.88 MB)
   10. Data_s3_107.csv (765.78 MB)
   ... and 103 more files

STAGE 3.2 COMPLETE - Soft labels added to all files!


In [9]:
# ============================================================================
# STAGE 3.2: VERIFICATION - Load and Check Output
# ============================================================================

print("\n" + "=" * 100)
print("VERIFICATION: Checking Soft-Labeled Data")
print("=" * 100)

# Load first output file
output_files = sorted(STAGE3_OUTPUT_DIR.glob("*.csv"))

if len(output_files) > 0:
    sample_file = output_files[0]
    
    print(f"\n[INFO] Loading sample file: {sample_file.name}")
    
    df_sample = pd.read_csv(sample_file)
    
    print(f"\nDataFrame Info:")
    print(f"   Rows: {len(df_sample):,}")
    print(f"   Columns: {len(df_sample.columns)}")
    
    print(f"\nColumn Names:")
    for i, col in enumerate(df_sample.columns, 1):
        print(f"   {i}. {col}")
    
    # Check topic columns
    topic_cols = ['War', 'Health', 'Technology', 'Climate', 'Economics']
    
    print(f"\nTopic Similarity Statistics:")
    print(f"{'Topic':<15} {'Mean':<10} {'Min':<10} {'Max':<10} {'Std'}")
    print("=" * 60)
    
    for topic in topic_cols:
        if topic in df_sample.columns:
            mean_val = df_sample[topic].mean()
            min_val = df_sample[topic].min()
            max_val = df_sample[topic].max()
            std_val = df_sample[topic].std()
            print(f"{topic:<15} {mean_val:<10.4f} {min_val:<10.4f} {max_val:<10.4f} {std_val:.4f}")
    
    # Show sample rows
    print(f"\nSample Rows (Topic Similarity Columns Only):")
    print(df_sample[topic_cols].head(10).to_string(index=False))
    
    # Show one full row
    print(f"\nFull Row Example (Row 0):")
    for col in df_sample.columns:
        value = df_sample[col].iloc[0]
        if col in topic_cols:
            print(f"   {col}: {value:.6f}")
        elif col in ['w3_embedding', 'w5_embedding']:
            print(f"   {col}: {str(value)[:80]}...")
        else:
            print(f"   {col}: {value}")

else:
    print("[ERROR] No output files found")

print("\n" + "=" * 100)
print("VERIFICATION COMPLETE")
print("=" * 100)


VERIFICATION: Checking Soft-Labeled Data

[INFO] Loading sample file: Data_s3_1.csv

DataFrame Info:
   Rows: 14,704
   Columns: 16

Column Names:
   1. sentence_id
   2. article_id
   3. date
   4. source
   5. previous_sentence_1
   6. previous_sentence_2
   7. main_sentence
   8. next_sentence_1
   9. next_sentence_2
   10. w3_embedding
   11. w5_embedding
   12. War
   13. Health
   14. Technology
   15. Climate
   16. Economics

Topic Similarity Statistics:
Topic           Mean       Min        Max        Std
War             0.1046     -0.1373    0.5634     0.1020
Health          0.0910     -0.1497    0.4170     0.0721
Technology      0.0799     -0.1247    0.4656     0.0719
Climate         0.0878     -0.1247    0.6778     0.0809
Economics       0.1109     -0.1319    0.5369     0.0981

Sample Rows (Topic Similarity Columns Only):
     War   Health  Technology  Climate  Economics
0.021241 0.045396    0.043504 0.049677   0.079809
0.022722 0.028694    0.076324 0.005969  -0.024541
0.0

In [1]:
# ============================================================================
# STAGE 3.3: IMPORTS & PATH SETUP
# ============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re
from IPython.display import clear_output

print("✅ Stage 3.3 imports loaded successfully!")
print(f"   - pandas: {pd.__version__}")
print(f"   - numpy: {np.__version__}")
print(f"   - pathlib, re, IPython.display")

# Define base directory path
PROCESSED_DATA_DIR = Path("../Processed_Data")

# Define paths for Stage 3.3
STAGE3_INPUT_DIR = PROCESSED_DATA_DIR / "Stage_3_w5"
STAGE33_OUTPUT_DIR = PROCESSED_DATA_DIR / "Stage_3_3_w5"
TOPIC_WISE_OUTPUT_DIR = PROCESSED_DATA_DIR / "Topic_Wise"

print(f"\n📁 Paths configured:")
print(f"   Base:   {PROCESSED_DATA_DIR.absolute()}")
print(f"   Input:  {STAGE3_INPUT_DIR}")
print(f"   Stage 3.3: {STAGE33_OUTPUT_DIR}")
print(f"   Topic-Wise: {TOPIC_WISE_OUTPUT_DIR}")

✅ Stage 3.3 imports loaded successfully!
   - pandas: 3.0.1
   - numpy: 2.4.2
   - pathlib, re, IPython.display

📁 Paths configured:
   Base:   /home/hp/SEM2/INLP/Naretve_Shift/Pre_Processin/../Processed_Data
   Input:  ../Processed_Data/Stage_3_w5
   Stage 3.3: ../Processed_Data/Stage_3_3_w5
   Topic-Wise: ../Processed_Data/Topic_Wise


---

# 🎯 STAGE 3.3: Topic-Wise Filtering & Aggregation

## Overview
This stage filters rows based on topic similarity thresholds and creates topic-specific datasets.

## Process Flow:
1. **Filter by Threshold**: For each file, extract rows where topic score > threshold
2. **Organize by Topic**: Save filtered rows into topic-specific folders
3. **Combine Topic Data**: Merge all files per topic into single CSV
4. **Final Output**: 5 topic-wise CSV files with only `date` and `w5_embedding` columns

## Output Structure:
```
Processed_Data/
├── Stage_3_3/           # Intermediate filtered files per topic
│   ├── War/
│   │   ├── Data_S33_1.csv
│   │   └── Data_S33_2.csv
│   ├── Health/
│   ├── Technology/
│   ├── Climate/
│   └── Economics/
└── Topic_Wise/          # Final combined topic files
    ├── War.csv          (date, w5_embedding)
    ├── Health.csv
    ├── Technology.csv
    ├── Climate.csv
    └── Economics.csv
```

---

## Step 3.3.1: Configuration & Setup

In [2]:
# ============================================================================
# STAGE 3.3: CONFIGURATION & DIRECTORY SETUP
# ============================================================================

print("\n" + "=" * 100)
print("STAGE 3.3: TOPIC-WISE FILTERING & AGGREGATION - CONFIGURATION")
print("=" * 100)

# Define paths
STAGE3_INPUT_DIR = PROCESSED_DATA_DIR / "Stage_3_w5"
STAGE33_OUTPUT_DIR = PROCESSED_DATA_DIR / "Stage_3_3_w5"
TOPIC_WISE_OUTPUT_DIR = PROCESSED_DATA_DIR / "Topic_Wise"

# Topic list
TOPICS = ['War', 'Health', 'Technology', 'Climate', 'Economics']

# Threshold for topic filtering (adjustable)
TOPIC_THRESHOLD = 0.3  # Rows with topic score > 0.3 will be included

print(f"\n[CONFIG] Stage 3.3 Settings:")
print(f"   Input Directory:     {STAGE3_INPUT_DIR}")
print(f"   Stage 3.3 Output:    {STAGE33_OUTPUT_DIR}")
print(f"   Final Output:        {TOPIC_WISE_OUTPUT_DIR}")
print(f"   Topic Threshold:     {TOPIC_THRESHOLD}")
print(f"   Topics:              {', '.join(TOPICS)}")

# Create output directories
STAGE33_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOPIC_WISE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create topic-specific folders
for topic in TOPICS:
    topic_dir = STAGE33_OUTPUT_DIR / topic
    topic_dir.mkdir(parents=True, exist_ok=True)
    print(f"   ✓ Created: {topic_dir.name}/")

print(f"\n✅ Directory structure created successfully!")

# Verify input files
input_files = sorted(STAGE3_INPUT_DIR.glob("Data_s3_*.csv"))
print(f"\n[INFO] Found {len(input_files)} files in Stage 3 to process")
if len(input_files) > 0:
    print(f"   First file: {input_files[0].name}")
    print(f"   Last file:  {input_files[-1].name}")
else:
    print(f"   ⚠️  No Data_s3_*.csv files found in {STAGE3_INPUT_DIR}")
    print(f"   Please run Stage 3.2 first!")


STAGE 3.3: TOPIC-WISE FILTERING & AGGREGATION - CONFIGURATION

[CONFIG] Stage 3.3 Settings:
   Input Directory:     ../Processed_Data/Stage_3_w5
   Stage 3.3 Output:    ../Processed_Data/Stage_3_3_w5
   Final Output:        ../Processed_Data/Topic_Wise
   Topic Threshold:     0.3
   Topics:              War, Health, Technology, Climate, Economics
   ✓ Created: War/
   ✓ Created: Health/
   ✓ Created: Technology/
   ✓ Created: Climate/
   ✓ Created: Economics/

✅ Directory structure created successfully!

[INFO] Found 0 files in Stage 3 to process
   ⚠️  No Data_s3_*.csv files found in ../Processed_Data/Stage_3_w5
   Please run Stage 3.2 first!


## Step 3.3.2: Filter Rows by Topic Threshold

Process each file and extract rows where topic score exceeds threshold. One row can belong to multiple topics.

In [8]:
# ============================================================================
# STAGE 3.3.2: FILTER ROWS BY TOPIC THRESHOLD - MULTI-THREADED
# ============================================================================

import threading
from queue import Queue
import time

print("\n" + "=" * 100)
print("STAGE 3.3.2: FILTERING ROWS BY TOPIC THRESHOLD (MULTI-THREADED)")
print("=" * 100)

# Configuration
MAX_WORKERS = 4  # Number of worker threads

# Thread-safe statistics
stats_lock = threading.Lock()
stats_s33 = {
    'total_files': len(input_files),
    'processed_files': 0,
    'failed_files': 0,
    'total_input_rows': 0,
    'topic_stats': {topic: {'files': 0, 'rows': 0} for topic in TOPICS},
    'processing': {},  # {thread_id: (filename, progress%)}
    'completed': []  # List of (filename, status, time)
}

start_time = time.time()


def process_single_file_s33(csv_file, thread_id):
    """Process a single file and filter by topic threshold"""
    
    file_start = time.time()
    
    try:
        # Update status
        with stats_lock:
            stats_s33['processing'][thread_id] = (csv_file.name, 0)
        
        # Extract file number from filename: Data_s3_107.csv -> 107
        match = re.search(r'Data_s3_(\d+)\.csv', csv_file.name, re.IGNORECASE)
        if match:
            file_num = match.group(1)
        else:
            with stats_lock:
                file_num = stats_s33['processed_files'] + 1
        
        # Load CSV (10% progress)
        with stats_lock:
            stats_s33['processing'][thread_id] = (csv_file.name, 10)
        
        df = pd.read_csv(csv_file)
        
        with stats_lock:
            stats_s33['total_input_rows'] += len(df)
            stats_s33['processing'][thread_id] = (csv_file.name, 20)
        
        # Check if topic columns exist
        missing_topics = [topic for topic in TOPICS if topic not in df.columns]
        if missing_topics:
            with stats_lock:
                stats_s33['failed_files'] += 1
                if thread_id in stats_s33['processing']:
                    del stats_s33['processing'][thread_id]
                file_time = time.time() - file_start
                stats_s33['completed'].append((csv_file.name, 'MISSING_COLS', file_time))
            return False
        
        # Filter and save for each topic (20% -> 90% progress)
        topic_row_counts = {}
        progress = 20
        
        for i, topic in enumerate(TOPICS):
            # Filter rows where topic score > threshold
            topic_df = df[df[topic] > TOPIC_THRESHOLD].copy()
            
            topic_row_counts[topic] = len(topic_df)
            
            if len(topic_df) > 0:
                # Save to topic-specific folder with Data_S33_i naming
                output_file = STAGE33_OUTPUT_DIR / topic / f"Data_S33_{file_num}.csv"
                topic_df.to_csv(output_file, index=False)
                
                with stats_lock:
                    stats_s33['topic_stats'][topic]['files'] += 1
                    stats_s33['topic_stats'][topic]['rows'] += len(topic_df)
            
            # Update progress
            progress = 20 + int((i + 1) / len(TOPICS) * 70)
            with stats_lock:
                stats_s33['processing'][thread_id] = (csv_file.name, progress)
        
        # Complete (100% progress)
        with stats_lock:
            stats_s33['processing'][thread_id] = (csv_file.name, 100)
            stats_s33['processed_files'] += 1
            file_time = time.time() - file_start
            stats_s33['completed'].append((csv_file.name, 'OK', file_time))
            if thread_id in stats_s33['processing']:
                del stats_s33['processing'][thread_id]
        
        return True
        
    except Exception as e:
        with stats_lock:
            stats_s33['failed_files'] += 1
            file_time = time.time() - file_start
            stats_s33['completed'].append((csv_file.name, f'ERROR: {str(e)[:30]}', file_time))
            if thread_id in stats_s33['processing']:
                del stats_s33['processing'][thread_id]
        return False


def worker_s33(file_queue, thread_id):
    """Worker thread that processes files from queue"""
    
    while True:
        try:
            csv_file = file_queue.get(timeout=1)
            if csv_file is None:  # Poison pill
                break
            
            # Process file
            success = process_single_file_s33(csv_file, thread_id)
            
            file_queue.task_done()
            
        except:
            break


def display_progress_s33():
    """Display simple progress summary"""
    with stats_lock:
        completed = stats_s33['processed_files'] + stats_s33['failed_files']
        percent = (completed / stats_s33['total_files'] * 100) if stats_s33['total_files'] > 0 else 0
        elapsed = time.time() - start_time
        
        print(f"\n{'=' * 80}")
        print(f"PROGRESS: {completed}/{stats_s33['total_files']} files ({percent:.1f}%)")
        print(f"Elapsed: {elapsed:.0f}s | OK: {stats_s33['processed_files']} | Failed: {stats_s33['failed_files']}")
        print(f"{'=' * 80}")
        
        # Show active threads
        if stats_s33['processing']:
            print(f"\nActive Threads:")
            for tid, (fname, progress) in stats_s33['processing'].items():
                bar_len = int(progress / 5)
                bar = '█' * bar_len + '░' * (20 - bar_len)
                print(f"  T{tid:2d}: [{bar}] {progress:3d}% | {fname}")
        
        # Show last 5 completed
        if stats_s33['completed']:
            print(f"\nRecently Completed (last 5):")
            for fname, status, ftime in stats_s33['completed'][-5:]:
                print(f"  [{status:6s}] {fname} ({ftime:.1f}s)")
        
        print(f"{'=' * 80}\n")


# Only proceed if there are files to process
if len(input_files) == 0:
    print(f"\n{'=' * 100}")
    print(f"⚠️  NO FILES TO PROCESS")
    print(f"{'=' * 100}")
else:
    # Create queue and add files
    file_queue = Queue()
    for csv_file in input_files:
        file_queue.put(csv_file)

    # Start workers
    print(f"\n[CONFIG] Worker threads: {MAX_WORKERS}")
    print(f"[CONFIG] Files to process: {len(input_files)}")
    print(f"[CONFIG] Topic threshold: {TOPIC_THRESHOLD}")
    print(f"\n[INFO] Starting {MAX_WORKERS} worker threads...\n")
    
    threads = []

    for i in range(MAX_WORKERS):
        t = threading.Thread(target=worker_s33, args=(file_queue, i+1))
        t.daemon = True
        t.start()
        threads.append(t)

    # Monitor progress
    try:
        last_display = 0
        
        while True:
            # Display progress every 10 seconds
            current_time = time.time()
            if current_time - last_display >= 10:
                clear_output(wait=True)
                display_progress_s33()
                last_display = current_time
            
            # Check if all done
            if file_queue.empty() and all(not t.is_alive() for t in threads):
                break
            
            time.sleep(1)
            
    except KeyboardInterrupt:
        print("\n\n[WARN] Interrupted by user")

    # Wait for completion
    file_queue.join()

    # Send poison pills
    for _ in range(MAX_WORKERS):
        file_queue.put(None)

    # Wait for threads to finish
    for t in threads:
        t.join()

    elapsed_time = time.time() - start_time

    # Final display
    clear_output(wait=True)
    print(f"\n{'=' * 100}")
    print(f"STAGE 3.3.2 FILTERING COMPLETE")
    print(f"Time elapsed: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")
    if len(input_files) > 0:
        print(f"Speed: {len(input_files) / (elapsed_time/60):.2f} files/minute")
    print(f"{'=' * 100}")

    print(f"\nOverall Statistics:")
    print(f"   Total files processed:  {stats_s33['processed_files']}")
    print(f"   Total input rows:       {stats_s33['total_input_rows']:,}")
    print(f"   Threshold used:         {TOPIC_THRESHOLD}")

    print(f"\nPer-Topic Statistics:")
    print(f"   {'Topic':<15} {'Files':<10} {'Rows':<15} {'Avg Rows/File'}")
    print(f"   {'-'*15} {'-'*10} {'-'*15} {'-'*15}")

    for topic in TOPICS:
        files = stats_s33['topic_stats'][topic]['files']
        rows = stats_s33['topic_stats'][topic]['rows']
        avg = rows / files if files > 0 else 0
        print(f"   {topic:<15} {files:<10} {rows:<15,} {avg:,.1f}")

    print("\n" + "=" * 100)


STAGE 3.3.2 FILTERING COMPLETE
Time elapsed: 1840.46 seconds (30.67 minutes)
Speed: 3.68 files/minute

Overall Statistics:
   Total files processed:  113
   Total input rows:       3,237,195
   Threshold used:         0.3

Per-Topic Statistics:
   Topic           Files      Rows            Avg Rows/File
   --------------- ---------- --------------- ---------------
   War             113        497,224         4,400.2
   Health          113        190,235         1,683.5
   Technology      113        191,615         1,695.7
   Climate         113        189,483         1,676.8
   Economics       113        280,208         2,479.7



## Step 3.3.3: Combine Topic Files

Merge all files for each topic into a single CSV with `date`, `w5_embedding`, and `main_sentence` columns.

In [9]:
# ============================================================================
# STAGE 3.3.3: COMBINE TOPIC FILES INTO FINAL CSV
# ============================================================================

print("\n" + "=" * 100)
print("STAGE 3.3.3: COMBINING TOPIC FILES")
print("=" * 100)

# Required columns for final output
REQUIRED_COLUMNS = ['date', 'w5_embedding','main_sentence','War', 'Health', 'Technology', 'Climate', 'Economics']

combine_stats = {}

for topic in TOPICS:
    print(f"\n[TOPIC] Processing: {topic}")
    
    # Get all Data_S33_*.csv files for this topic
    topic_folder = STAGE33_OUTPUT_DIR / topic
    topic_files = sorted(topic_folder.glob("Data_S33_*.csv"))
    
    print(f"   Found {len(topic_files)} files")
    
    if len(topic_files) == 0:
        print(f"   ⚠️  No files found - Skipping")
        combine_stats[topic] = {'files': 0, 'rows': 0}
        continue
    
    # Combine all files for this topic
    combined_dfs = []
    
    for topic_file in topic_files:
        df = pd.read_csv(topic_file)
        
        # Check if required columns exist
        missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
        if missing_cols:
            print(f"   ⚠️  {topic_file.name} missing columns: {missing_cols} - Skipping")
            continue
        
        # Select only required columns
        df_selected = df[REQUIRED_COLUMNS].copy()
        combined_dfs.append(df_selected)
    
    if len(combined_dfs) == 0:
        print(f"   ⚠️  No valid data found - Skipping")
        combine_stats[topic] = {'files': 0, 'rows': 0}
        continue
    
    # Concatenate all dataframes
    combined_df = pd.concat(combined_dfs, ignore_index=True)
    
    # Save to Topic_Wise folder
    output_file = TOPIC_WISE_OUTPUT_DIR / f"{topic}.csv"
    combined_df.to_csv(output_file, index=False)
    
    combine_stats[topic] = {
        'files': len(topic_files),
        'rows': len(combined_df)
    }
    
    print(f"   ✅ Combined {len(topic_files)} files → {len(combined_df):,} rows")
    print(f"   Saved to: {output_file.name}")

print("\n" + "=" * 100)
print("COMBINATION COMPLETE - FINAL STATISTICS")
print("=" * 100)

print(f"\nFinal Topic-Wise CSV Files:")
print(f"   {'Topic':<15} {'Source Files':<15} {'Total Rows':<15} {'Output File'}")
print(f"   {'-'*15} {'-'*15} {'-'*15} {'-'*30}")

for topic in TOPICS:
    if topic in combine_stats:
        files = combine_stats[topic]['files']
        rows = combine_stats[topic]['rows']
        output_name = f"{topic}.csv"
        print(f"   {topic:<15} {files:<15} {rows:<15,} {output_name}")

print(f"\n📁 Output Location: {TOPIC_WISE_OUTPUT_DIR}")
print(f"   Columns in each file: {', '.join(REQUIRED_COLUMNS)}")

print("\n" + "=" * 100)
print("✅ STAGE 3.3 COMPLETE - All topic-wise CSV files created!")
print("=" * 100)


STAGE 3.3.3: COMBINING TOPIC FILES

[TOPIC] Processing: War
   Found 113 files
   ✅ Combined 113 files → 497,224 rows
   Saved to: War.csv

[TOPIC] Processing: Health
   Found 113 files
   ✅ Combined 113 files → 190,235 rows
   Saved to: Health.csv

[TOPIC] Processing: Technology
   Found 113 files
   ✅ Combined 113 files → 191,615 rows
   Saved to: Technology.csv

[TOPIC] Processing: Climate
   Found 113 files
   ✅ Combined 113 files → 189,483 rows
   Saved to: Climate.csv

[TOPIC] Processing: Economics
   Found 113 files
   ✅ Combined 113 files → 280,208 rows
   Saved to: Economics.csv

COMBINATION COMPLETE - FINAL STATISTICS

Final Topic-Wise CSV Files:
   Topic           Source Files    Total Rows      Output File
   --------------- --------------- --------------- ------------------------------
   War             113             497,224         War.csv
   Health          113             190,235         Health.csv
   Technology      113             191,615         Technology.csv
  

## Step 3.3.4: Verification

Verify the final topic-wise CSV files.

In [ ]:
# ============================================================================
# STAGE 3.3.4: VERIFICATION
# ============================================================================

print("\n" + "=" * 100)
print("STAGE 3.3: VERIFICATION OF TOPIC-WISE FILES")
print("=" * 100)

# Check final output files
output_files = sorted(TOPIC_WISE_OUTPUT_DIR.glob("*.csv"))

if len(output_files) == 0:
    print("\n⚠️  No output files found!")
else:
    print(f"\n✅ Found {len(output_files)} topic-wise CSV files")
    
    print(f"\n{'='*100}")
    print(f"FILE DETAILS:")
    print(f"{'='*100}")
    
    for output_file in output_files:
        # Load file
        df = pd.read_csv(output_file)
        file_size_mb = output_file.stat().st_size / (1024 * 1024)
        
        print(f"\n📄 {output_file.name}")
        print(f"   File size:      {file_size_mb:.2f} MB")
        print(f"   Rows:           {len(df):,}")
        print(f"   Columns:        {list(df.columns)}")
        
        # Check for required columns
        if 'date' in df.columns and 'w5_embedding' in df.columns:
            print(f"   ✅ Required columns present")
            
            # Show date range if available
            if not df['date'].isna().all():
                try:
                    dates = pd.to_datetime(df['date'], errors='coerce')
                    valid_dates = dates.dropna()
                    if len(valid_dates) > 0:
                        print(f"   Date range:     {valid_dates.min()} to {valid_dates.max()}")
                except:
                    pass
            
            # Check embedding column
            non_null_embeddings = df['w5_embedding'].notna().sum()
            print(f"   Valid embeddings: {non_null_embeddings:,} ({non_null_embeddings/len(df)*100:.1f}%)")
        else:
            print(f"   ⚠️  Missing required columns!")
    
    # Show sample from first file
    if len(output_files) > 0:
        print(f"\n{'='*100}")
        print(f"SAMPLE DATA (from {output_files[0].name}):")
        print(f"{'='*100}")
        
        df_sample = pd.read_csv(output_files[0])
        print(f"\nFirst 5 rows:")
        
        # Display with truncated embedding
        df_display = df_sample.head().copy()
        if 'w5_embedding' in df_display.columns:
            df_display['w5_embedding'] = df_display['w5_embedding'].astype(str).str[:60] + '...'
        
        print(df_display.to_string(index=False))

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE")
print("=" * 100)


STAGE 3.3: VERIFICATION OF TOPIC-WISE FILES

✅ Found 5 topic-wise CSV files

FILE DETAILS:

📄 Climate.csv
   File size:      1761.45 MB
   Rows:           188,013
   Columns:        ['date', 'w5_embedding', 'main_sentence', 'War', 'Health', 'Technology', 'Climate', 'Economics']
   ✅ Required columns present
   Date range:     2011-09-21 13:59:11 to 2025-10-26 10:38:15
   Valid embeddings: 188,013 (100.0%)

📄 Climate.csv
   File size:      1761.45 MB
   Rows:           188,013
   Columns:        ['date', 'w5_embedding', 'main_sentence', 'War', 'Health', 'Technology', 'Climate', 'Economics']
   ✅ Required columns present
   Date range:     2011-09-21 13:59:11 to 2025-10-26 10:38:15
   Valid embeddings: 188,013 (100.0%)

📄 Economics.csv
   File size:      2602.14 MB
   Rows:           277,886
   Columns:        ['date', 'w5_embedding', 'main_sentence', 'War', 'Health', 'Technology', 'Climate', 'Economics']
   ✅ Required columns present
   Date range:     2011-09-21 16:29:11 to 2025-10-26

## Step 3.3.5: Data Analysis & Visualization

Analyze topic distribution over time and check for duplicate records.

In [1]:
# ============================================================================
# STAGE 3.3.5: DATA ANALYSIS & VISUALIZATION
# ============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

print("\n" + "=" * 100)
print("STAGE 3.3.5: DATA ANALYSIS & VISUALIZATION")
print("=" * 100)

# Load all topic CSV files
topic_data = {}

for topic in TOPICS:
    topic_file = TOPIC_WISE_OUTPUT_DIR / f"{topic}.csv"
    
    if topic_file.exists():
        df = pd.read_csv(topic_file)
        topic_data[topic] = df
        print(f"\n✅ Loaded: {topic}.csv ({len(df):,} rows)")
    else:
        print(f"\n⚠️  Not found: {topic}.csv")

# ============================================================================
# 1. CHECK FOR DUPLICATES
# ============================================================================

print("\n" + "=" * 100)
print("DUPLICATE ANALYSIS")
print("=" * 100)

duplicate_summary = []

for topic, df in topic_data.items():
    total_rows = len(df)
    
    # Check for duplicates based on all columns
    duplicates_all = df.duplicated().sum()
    
    # Check for duplicates based on main_sentence only
    duplicates_sentence = df.duplicated(subset=['main_sentence']).sum() if 'main_sentence' in df.columns else 0
    
    # Check for duplicates based on date + main_sentence
    duplicates_date_sentence = df.duplicated(subset=['date', 'main_sentence']).sum() if 'date' in df.columns and 'main_sentence' in df.columns else 0
    
    duplicate_summary.append({
        'Topic': topic,
        'Total Rows': total_rows,
        'All Columns Dup': duplicates_all,
        'Sentence Dup': duplicates_sentence,
        'Date+Sentence Dup': duplicates_date_sentence
    })
    
    print(f"\n📊 {topic}:")
    print(f"   Total rows:                {total_rows:,}")
    print(f"   Duplicates (all columns):  {duplicates_all:,} ({duplicates_all/total_rows*100:.2f}%)")
    print(f"   Duplicates (sentence):     {duplicates_sentence:,} ({duplicates_sentence/total_rows*100:.2f}%)")
    print(f"   Duplicates (date+sentence): {duplicates_date_sentence:,} ({duplicates_date_sentence/total_rows*100:.2f}%)")

# Create summary DataFrame
dup_df = pd.DataFrame(duplicate_summary)
print(f"\n{'='*100}")
print("DUPLICATE SUMMARY TABLE:")
print(f"{'='*100}")
print(dup_df.to_string(index=False))

# ============================================================================
# 2. DATA DISTRIBUTION OVER TIME
# ============================================================================

print("\n" + "=" * 100)
print("TEMPORAL DISTRIBUTION ANALYSIS")
print("=" * 100)

# Prepare data for plotting
temporal_data = {}

for topic, df in topic_data.items():
    if 'date' in df.columns:
        # Convert date column to datetime
        df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
        
        # Remove rows with invalid dates
        df_valid = df[df['date_parsed'].notna()].copy()
        
        if len(df_valid) > 0:
            temporal_data[topic] = df_valid
            
            # Show date range
            min_date = df_valid['date_parsed'].min()
            max_date = df_valid['date_parsed'].max()
            print(f"\n📅 {topic}:")
            print(f"   Valid dates: {len(df_valid):,} / {len(df):,} ({len(df_valid)/len(df)*100:.1f}%)")
            print(f"   Date range:  {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
            print(f"   Duration:    {(max_date - min_date).days} days")
        else:
            print(f"\n⚠️  {topic}: No valid dates found")
    else:
        print(f"\n⚠️  {topic}: No date column found")

# ============================================================================
# 3. VISUALIZATION: TOPIC DISTRIBUTION OVER TIME
# ============================================================================

if len(temporal_data) > 0:
    print("\n" + "=" * 100)
    print("GENERATING VISUALIZATIONS...")
    print("=" * 100)
    
    # Create figure with subplots
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    fig.suptitle('Topic Distribution Over Time', fontsize=16, fontweight='bold')
    
    # Flatten axes for easier iteration
    axes_flat = axes.flatten()
    
    for idx, (topic, df) in enumerate(temporal_data.items()):
        ax = axes_flat[idx]
        
        # Group by date and count
        daily_counts = df.groupby(df['date_parsed'].dt.date).size()
        
        # Plot
        ax.plot(daily_counts.index, daily_counts.values, marker='o', linestyle='-', markersize=3, linewidth=1.5)
        ax.set_title(f'{topic} ({len(df):,} records)', fontsize=12, fontweight='bold')
        ax.set_xlabel('Date', fontsize=10)
        ax.set_ylabel('Number of Records', fontsize=10)
        ax.grid(True, alpha=0.3)
        
        # Format x-axis
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        ax.tick_params(axis='x', rotation=45)
        
        # Add statistics
        avg_count = daily_counts.mean()
        max_count = daily_counts.max()
        ax.axhline(y=avg_count, color='r', linestyle='--', alpha=0.5, label=f'Avg: {avg_count:.1f}')
        ax.legend(fontsize=8)
    
    # Hide the 6th subplot if we only have 5 topics
    if len(temporal_data) < 6:
        axes_flat[5].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ Visualization complete!")
    
    # ============================================================================
    # 4. COMBINED TIMELINE PLOT
    # ============================================================================
    
    print("\n" + "=" * 100)
    print("GENERATING COMBINED TIMELINE...")
    print("=" * 100)
    
    plt.figure(figsize=(16, 8))
    
    for topic, df in temporal_data.items():
        # Group by date and count
        daily_counts = df.groupby(df['date_parsed'].dt.date).size()
        
        # Plot
        plt.plot(daily_counts.index, daily_counts.values, marker='o', linestyle='-', 
                markersize=3, linewidth=2, label=f'{topic} ({len(df):,})', alpha=0.7)
    
    plt.title('All Topics - Temporal Distribution', fontsize=16, fontweight='bold')
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Number of Records', fontsize=12)
    plt.legend(fontsize=10, loc='best')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.tight_layout()
    plt.show()
    
    print("\n✅ Combined timeline complete!")
    
    # ============================================================================
    # 5. TOPIC VOLUME COMPARISON
    # ============================================================================
    
    print("\n" + "=" * 100)
    print("TOPIC VOLUME COMPARISON")
    print("=" * 100)
    
    # Create bar chart
    topic_counts = {topic: len(df) for topic, df in topic_data.items()}
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(topic_counts.keys(), topic_counts.values(), color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
    
    plt.title('Total Records per Topic', fontsize=16, fontweight='bold')
    plt.xlabel('Topic', fontsize=12)
    plt.ylabel('Number of Records', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n✅ Volume comparison complete!")

else:
    print("\n⚠️  No temporal data available for visualization")

print("\n" + "=" * 100)
print("✅ ANALYSIS & VISUALIZATION COMPLETE")
print("=" * 100)


STAGE 3.3.5: DATA ANALYSIS & VISUALIZATION


NameError: name 'TOPICS' is not defined

## Step 3.3.6: Remove Duplicates

Remove duplicate rows (based on all columns) from each topic CSV file and resave.

In [ ]:
# ============================================================================
# STAGE 3.3.6: REMOVE DUPLICATES FROM TOPIC FILES
# ============================================================================

# ===== IMPORT LIBRARIES =====
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ===== DEFINE ALL PATHS (STANDALONE - NO DEPENDENCIES) =====
# Automatically detect workspace directory
CURRENT_DIR = Path.cwd()
print(f"Current working directory: {CURRENT_DIR}\n")

# Determine base directory intelligently
if 'Pre_Processin' in str(CURRENT_DIR):
    # Running from Pre_Processin folder
    BASE_DIR = CURRENT_DIR.parent / "Processed_Data"
else:
    # Running from main project folder
    BASE_DIR = CURRENT_DIR / "Processed_Data"

# Define all required paths
TOPIC_WISE_OUTPUT_DIR = BASE_DIR / "Topic_Wise_w5"

# Create directory if it doesn't exist
TOPIC_WISE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== DEFINE TOPICS LIST =====
TOPICS = ['War', 'Health', 'Technology', 'Climate', 'Economics']

# Display configuration
print("=" * 100)
print("CONFIGURATION")
print("=" * 100)
print(f"Base Directory:        {BASE_DIR}")
print(f"Topic-wise Output:     {TOPIC_WISE_OUTPUT_DIR}")
print(f"Topics to Process:     {', '.join(TOPICS)}")

# ===== START DEDUPLICATION =====
print("\n" + "=" * 100)
print("STAGE 3.3.6: REMOVING DUPLICATES")
print("=" * 100)

dedup_stats = []

for topic in TOPICS:
    topic_file = TOPIC_WISE_OUTPUT_DIR / f"{topic}.csv"
    
    if not topic_file.exists():
        print(f"\n⚠️  {topic}.csv not found - Skipping")
        continue
    
    print(f"\n📄 Processing: {topic}.csv")
    
    # Load CSV
    df = pd.read_csv(topic_file)
    original_rows = len(df)
    print(f"   Original rows: {original_rows:,}")
    
    # Step 1: Remove duplicates based on date + main_sentence
    if 'date' in df.columns and 'main_sentence' in df.columns:
        df_dedup_step1 = df.drop_duplicates(subset=['date', 'main_sentence'], keep='first')
        removed_step1 = original_rows - len(df_dedup_step1)
        print(f"   Removed (date+sentence): {removed_step1:,}")
    else:
        df_dedup_step1 = df.copy()
        removed_step1 = 0
        print(f"   ⚠️  'date' or 'main_sentence' column missing - skipping date+sentence dedup")
    
    # Step 2: Remove duplicates based on all columns
    df_dedup_final = df_dedup_step1.drop_duplicates(keep='first')
    removed_step2 = len(df_dedup_step1) - len(df_dedup_final)
    print(f"   Removed (all columns):   {removed_step2:,}")
    
    final_rows = len(df_dedup_final)
    total_removed = original_rows - final_rows
    
    print(f"   Final rows:    {final_rows:,}")
    print(f"   Total removed: {total_removed:,} ({total_removed/original_rows*100:.2f}%)")
    
    # Save back to same file
    df_dedup_final.to_csv(topic_file, index=False)
    print(f"   ✅ Saved: {topic_file.name}")
    
    # Collect statistics
    dedup_stats.append({
        'Topic': topic,
        'Original Rows': original_rows,
        'Final Rows': final_rows,
        'Date+Sentence Dup': removed_step1,
        'All Columns Dup': removed_step2,
        'Total Removed': total_removed,
        'Removed %': f"{total_removed/original_rows*100:.2f}%"
    })

# Display summary table
print("\n" + "=" * 100)
print("DEDUPLICATION SUMMARY")
print("=" * 100)

if dedup_stats:
    dedup_df = pd.DataFrame(dedup_stats)
    print(f"\n{dedup_df.to_string(index=False)}")
    
    total_original = sum([s['Original Rows'] for s in dedup_stats])
    total_final = sum([s['Final Rows'] for s in dedup_stats])
    total_removed = total_original - total_final
    
    print(f"\n{'='*100}")
    print(f"TOTALS:")
    print(f"   Original rows: {total_original:,}")
    print(f"   Final rows:    {total_final:,}")
    print(f"   Removed:       {total_removed:,} ({total_removed/total_original*100:.2f}%)")
else:
    print("\n⚠️  No files processed")

print("\n" + "=" * 100)
print("✅ DEDUPLICATION COMPLETE - All files updated!")
print("=" * 100)



STAGE 3.3.6: REMOVING DUPLICATES

📄 Processing: War.csv
   Original rows: 497,224


In [ ]:
# ============================================================================
# STAGE 3.3.7: FINAL DATA VERIFICATION
# ============================================================================

# ===== IMPORT LIBRARIES =====
import pandas as pd
import numpy as np
from pathlib import Path
import gc
import warnings
warnings.filterwarnings('ignore')

# ===== DEFINE ALL PATHS (STANDALONE - NO DEPENDENCIES) =====
# Automatically detect workspace directory
CURRENT_DIR = Path.cwd()
print(f"Current working directory: {CURRENT_DIR}\n")

# Determine base directory intelligently
if 'Pre_Processin' in str(CURRENT_DIR):
    # Running from Pre_Processin folder
    BASE_DIR = CURRENT_DIR.parent / "Processed_Data"
else:
    # Running from main project folder
    BASE_DIR = CURRENT_DIR / "Processed_Data"

# Define all required paths
TOPIC_WISE_OUTPUT_DIR = BASE_DIR / "Topic_Wise_w5"
OUTPUT_DIR = CURRENT_DIR.parent / "Window_Comparison_Results" if 'Pre_Processin' in str(CURRENT_DIR) else CURRENT_DIR / "Window_Comparison_Results"

# Create directories if they don't exist
TOPIC_WISE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== DEFINE TOPICS LIST =====
TOPICS = ['War', 'Health', 'Technology', 'Climate', 'Economics']

# Display configuration
print("=" * 100)
print("CONFIGURATION")
print("=" * 100)
print(f"Base Directory:        {BASE_DIR}")
print(f"Topic-wise Output:     {TOPIC_WISE_OUTPUT_DIR}")
print(f"Results Output:        {OUTPUT_DIR}")
print(f"Topics to Process:     {', '.join(TOPICS)}")

# ===== START VERIFICATION =====
print("\n" + "=" * 100)
print("STAGE 3.3.7: FINAL DATA VERIFICATION")
print("=" * 100)

verification_results = []

for topic in TOPICS:
    topic_file = TOPIC_WISE_OUTPUT_DIR / f"{topic}.csv"
    
    if not topic_file.exists():
        print(f"\n⚠️  {topic}.csv not found - Skipping")
        continue
    
    print(f"\n📊 Verifying: {topic}.csv")
    
    # Load file
    df = pd.read_csv(topic_file)
    
    # Basic statistics
    total_rows = len(df)
    total_cols = len(df.columns)
    file_size_mb = topic_file.stat().st_size / (1024 * 1024)
    
    print(f"   Total Rows: {total_rows:,}")
    print(f"   Total Columns: {total_cols}")
    print(f"   File Size: {file_size_mb:.2f} MB")
    
    # Check for missing values
    missing_count = df.isnull().sum().sum()
    missing_pct = (missing_count / (total_rows * total_cols)) * 100
    print(f"   Missing Values: {missing_count:,} ({missing_pct:.2f}%)")
    
    # Check date column
    if 'date' in df.columns:
        valid_dates = pd.to_datetime(df['date'], errors='coerce').notna().sum()
        print(f"   Valid Dates: {valid_dates:,} / {total_rows:,} ({valid_dates/total_rows*100:.2f}%)")
        
        # Date range
        df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
        date_range = f"{df['date_parsed'].min()} to {df['date_parsed'].max()}"
        print(f"   Date Range: {date_range}")
    else:
        valid_dates = 0
        date_range = "N/A"
        print(f"   ⚠️  'date' column not found")
    
    # Check topic similarity score
    if topic in df.columns:
        topic_mean = df[topic].mean()
        topic_std = df[topic].std()
        topic_min = df[topic].min()
        topic_max = df[topic].max()
        print(f"   Topic Score '{topic}': Mean={topic_mean:.4f}, Std={topic_std:.4f}, Range=[{topic_min:.4f}, {topic_max:.4f}]")
    else:
        topic_mean = topic_std = topic_min = topic_max = None
        print(f"   ⚠️  Topic column '{topic}' not found")
    
    # Check embedding column
    if 'embedding' in df.columns:
        # Sample one embedding to check format
        sample_emb = str(df['embedding'].iloc[0])
        emb_preview = sample_emb[:100] + "..." if len(sample_emb) > 100 else sample_emb
        print(f"   Embedding Format: {emb_preview}")
    else:
        print(f"   ⚠️  'embedding' column not found")
    
    # Check for duplicates (should be 0 after dedup)
    duplicates = df.duplicated().sum()
    print(f"   Duplicate Rows: {duplicates:,}")
    
    if duplicates > 0:
        print(f"   ⚠️  WARNING: Duplicates still exist!")
    else:
        print(f"   ✅ No duplicates")
    
    # Collect results
    verification_results.append({
        'Topic': topic,
        'Rows': total_rows,
        'Columns': total_cols,
        'File Size (MB)': f"{file_size_mb:.2f}",
        'Missing Values': missing_count,
        'Missing %': f"{missing_pct:.2f}%",
        'Valid Dates': valid_dates,
        'Date Range': date_range,
        'Topic Score Mean': f"{topic_mean:.4f}" if topic_mean else "N/A",
        'Duplicates': duplicates
    })
    
    # Cleanup
    del df
    gc.collect()

# Display verification summary
print("\n" + "=" * 100)
print("VERIFICATION SUMMARY")
print("=" * 100)

if verification_results:
    verify_df = pd.DataFrame(verification_results)
    print(f"\n{verify_df.to_string(index=False)}")
    
    # Overall totals
    total_rows = sum([int(r['Rows']) for r in verification_results])
    total_size = sum([float(r['File Size (MB)']) for r in verification_results])
    total_duplicates = sum([int(r['Duplicates']) for r in verification_results])
    
    print(f"\n{'='*100}")
    print(f"OVERALL TOTALS:")
    print(f"   Total Rows Across All Topics: {total_rows:,}")
    print(f"   Total Size: {total_size:.2f} MB")
    print(f"   Total Duplicates: {total_duplicates:,}")
    
    if total_duplicates == 0:
        print(f"   ✅ All files are clean (no duplicates)")
    else:
        print(f"   ⚠️  WARNING: {total_duplicates:,} duplicates found")
else:
    print("\n⚠️  No files found for verification")

print("\n" + "=" * 100)
print("✅ VERIFICATION COMPLETE")
print("=" * 100)


## Step 3.3.8: Final Summary Visualization

Create a comprehensive visualization showing the final state of all topic data.


In [ ]:
# ============================================================================
# STAGE 3.3.8: FINAL SUMMARY VISUALIZATION
# ============================================================================

# ===== IMPORT LIBRARIES =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ===== DEFINE ALL PATHS (STANDALONE - NO DEPENDENCIES) =====
# Automatically detect workspace directory
CURRENT_DIR = Path.cwd()

# Determine base directory intelligently
if 'Pre_Processin' in str(CURRENT_DIR):
    # Running from Pre_Processin folder
    BASE_DIR = CURRENT_DIR.parent / "Processed_Data"
    OUTPUT_DIR = CURRENT_DIR.parent / "Window_Comparison_Results"
else:
    # Running from main project folder
    BASE_DIR = CURRENT_DIR / "Processed_Data"
    OUTPUT_DIR = CURRENT_DIR / "Window_Comparison_Results"

# Define all required paths
TOPIC_WISE_OUTPUT_DIR = BASE_DIR / "Topic_Wise_w5"

# Create directories if they don't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== DEFINE TOPICS LIST =====
TOPICS = ['War', 'Health', 'Technology', 'Climate', 'Economics']

# NOTE: This cell expects 'verification_results' from previous cell (Cell 65)
# If running standalone, uncomment the following code to generate verification_results:

# verification_results = []
# for topic in TOPICS:
#     topic_file = TOPIC_WISE_OUTPUT_DIR / f"{topic}.csv"
#     if topic_file.exists():
#         df = pd.read_csv(topic_file)
#         verification_results.append({
#             'Topic': topic,
#             'Rows': len(df),
#             'File Size (MB)': f"{topic_file.stat().st_size / (1024 * 1024):.2f}",
#             'Missing %': f"{(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100):.2f}%"
#         })

# ===== START VISUALIZATION =====
print("\n" + "=" * 100)
print("STAGE 3.3.8: CREATING FINAL SUMMARY VISUALIZATION")
print("=" * 100)

if 'verification_results' in globals() and verification_results:
    # Prepare data for visualization
    topics = [r['Topic'] for r in verification_results]
    rows = [r['Rows'] for r in verification_results]
    file_sizes = [float(r['File Size (MB)']) for r in verification_results]
    missing_pcts = [float(r['Missing %'].replace('%', '')) for r in verification_results]
    
    # Create a 2x2 summary figure
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Final Topic Data Summary (After Cleaning & Deduplication)', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Plot 1: Row counts per topic
    ax1 = axes[0, 0]
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(topics)))
    bars1 = ax1.bar(topics, rows, color=colors, edgecolor='black', linewidth=1.5)
    ax1.set_title('Number of Rows per Topic', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Topic', fontsize=11)
    ax1.set_ylabel('Number of Rows', fontsize=11)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Plot 2: File sizes
    ax2 = axes[0, 1]
    bars2 = ax2.bar(topics, file_sizes, color=colors, edgecolor='black', linewidth=1.5)
    ax2.set_title('File Sizes (MB)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Topic', fontsize=11)
    ax2.set_ylabel('File Size (MB)', fontsize=11)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Plot 3: Missing value percentages
    ax3 = axes[1, 0]
    bars3 = ax3.bar(topics, missing_pcts, color=colors, edgecolor='black', linewidth=1.5)
    ax3.set_title('Missing Values (%)', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Topic', fontsize=11)
    ax3.set_ylabel('Missing Values (%)', fontsize=11)
    ax3.grid(axis='y', alpha=0.3, linestyle='--')
    ax3.set_ylim([0, max(missing_pcts) * 1.2 if max(missing_pcts) > 0 else 1])
    
    # Add value labels
    for bar in bars3:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Plot 4: Summary statistics table
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Create summary text
    total_rows = sum(rows)
    total_size = sum(file_sizes)
    avg_rows = np.mean(rows)
    avg_size = np.mean(file_sizes)
    avg_missing = np.mean(missing_pcts)
    
    summary_text = f"""
    📊 FINAL DATA SUMMARY
    {'='*40}
    
    Total Topics: {len(topics)}
    
    Total Rows: {total_rows:,}
    Total Size: {total_size:.2f} MB
    
    Average Rows/Topic: {avg_rows:,.0f}
    Average Size/Topic: {avg_size:.2f} MB
    Average Missing %: {avg_missing:.2f}%
    
    {'='*40}
    
    ✅ Data Cleaned
    ✅ Duplicates Removed
    ✅ Ready for Analysis
    
    Output Location:
    {TOPIC_WISE_OUTPUT_DIR}
    """
    
    ax4.text(0.1, 0.5, summary_text, 
            transform=ax4.transAxes,
            fontsize=11,
            verticalalignment='center',
            fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3, pad=1))
    
    plt.tight_layout()
    
    # Save figure
    summary_plot_path = OUTPUT_DIR / "FINAL_Data_Summary.png"
    plt.savefig(summary_plot_path, dpi=300, bbox_inches='tight')
    print(f"\n✅ Summary visualization saved: {summary_plot_path}")
    
    plt.show()
    
    # Also save summary as CSV
    summary_csv_path = OUTPUT_DIR / "FINAL_Data_Summary.csv"
    pd.DataFrame(verification_results).to_csv(summary_csv_path, index=False)
    print(f"✅ Summary CSV saved: {summary_csv_path}")
    
else:
    print("\n⚠️  No verification results found!")
    print("Please run Cell 65 (Final Data Verification) first to generate 'verification_results'")

print("\n" + "=" * 100)
print("✅ FINAL SUMMARY VISUALIZATION COMPLETE")
print("=" * 100)


## ✅ Data Preprocessing Complete!

### What Was Done:
1. ✅ **Data Loaded** - Combined multiple data sources
2. ✅ **Duplicates Removed** - Based on date+sentence and all columns
3. ✅ **Data Verified** - Checked for missing values, duplicates, and data quality
4. ✅ **Statistics Generated** - Created comprehensive summary

### Output Files:
```
Processed_Data/
├── Topic_Wise_w5/
│   ├── War.csv
│   ├── Health.csv
│   ├── Technology.csv
│   ├── Climate.csv
│   └── Economics.csv
└── Window_Comparison_Results/
    ├── FINAL_Data_Summary.png
    └── FINAL_Data_Summary.csv
```

### Next Steps:
1. **Review the summary visualization** to understand data distribution
2. **Check individual topic files** in `Topic_Wise_w5/` folder
3. **Use this cleaned data** for narrative shift detection or other analysis
4. **Run W3 vs W5 comparison** (cells 1-12 above) if you have both window sizes

### Key Metrics:
- **Total Topics**: 5 (War, Health, Technology, Climate, Economics)
- **Data Quality**: Deduplicated and verified
- **File Format**: CSV with embeddings ready for analysis


# 🔬 Topic Model Comparison: W3 vs W5 Embeddings

Compare window size 3 vs window size 5 to determine which produces better topic labeling.

**Analysis includes:**
1. Statistical metrics (mean, median, std of similarity scores)
2. Embedding compactness (intra-topic variance using cosine distance)
3. Overlap analysis (sentences appearing in multiple topics)
4. Final recommendation per topic based on multiple criteria

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from scipy.spatial.distance import cosine
from typing import Dict, List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION - WORKS ON BOTH KAGGLE AND LOCAL
# ============================================================================

# Detect if running on Kaggle
import os
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    print("🔵 Running on KAGGLE")
    # Kaggle paths
    W3_DIR = Path("/kaggle/input/your-dataset-w3/Topic_Wise_w3")  # UPDATE THIS PATH
    W5_DIR = Path("/kaggle/input/your-dataset-w5/Topic_Wise_w5")  # UPDATE THIS PATH
    OUTPUT_DIR = Path("/kaggle/working")
else:
    print("🟢 Running LOCALLY")
    # Local paths
    W3_DIR = Path("Processed_Data/Topic_Wise_w3")
    W5_DIR = Path("Processed_Data/Topic_Wise_w5")
    OUTPUT_DIR = Path("Processed_Data")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOPICS = ['War', 'Health', 'Technology', 'Climate', 'Economics']

print("=" * 100)
print("TOPIC MODEL COMPARISON: W3 vs W5 EMBEDDINGS")
print("=" * 100)
print(f"\nEnvironment: {'KAGGLE' if IS_KAGGLE else 'LOCAL'}")
print(f"W3 Directory: {W3_DIR}")
print(f"W5 Directory: {W5_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Topics: {', '.join(TOPICS)}")

# Verify directories exist
print(f"\n📁 Checking directories...")
if W3_DIR.exists():
    print(f"   ✅ W3 directory found: {W3_DIR}")
    w3_files = list(W3_DIR.glob("*.csv"))
    print(f"      Found {len(w3_files)} CSV files")
else:
    print(f"   ❌ W3 directory NOT found: {W3_DIR}")
    if IS_KAGGLE:
        print(f"      💡 Make sure you've added the W3 dataset to your Kaggle notebook")
        print(f"      💡 Update the path in the configuration cell above")

if W5_DIR.exists():
    print(f"   ✅ W5 directory found: {W5_DIR}")
    w5_files = list(W5_DIR.glob("*.csv"))
    print(f"      Found {len(w5_files)} CSV files")
else:
    print(f"   ❌ W5 directory NOT found: {W5_DIR}")
    if IS_KAGGLE:
        print(f"      💡 Make sure you've added the W5 dataset to your Kaggle notebook")
        print(f"      💡 Update the path in the configuration cell above")

print(f"\n{'-' * 100}\n")

TOPIC MODEL COMPARISON: W3 vs W5 EMBEDDINGS

W3 Directory: ../Processed_Data/Topic_Wise_w3
W5 Directory: ../Processed_Data/Topic_Wise_w5
Topics: War, Health, Technology, Climate, Economics
Output: Processed_Data

----------------------------------------------------------------------------------------------------



In [ ]:
# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def parse_embedding_string(emb_str):
    """Parse embedding string to numpy array.
    Handles both space-separated and comma-separated formats.
    """
    if pd.isna(emb_str):
        return None
    try:
        if isinstance(emb_str, str):
            # Remove quotes, brackets, and whitespace
            emb_str = emb_str.strip().strip('"').strip("'").strip('[]').strip()
            
            # Try comma-separated first (most common in CSV)
            if ',' in emb_str:
                values = [float(x.strip()) for x in emb_str.split(',') if x.strip()]
            else:
                # Fall back to space-separated
                values = [float(x) for x in emb_str.split() if x.strip()]
            
            return np.array(values) if values else None
        elif isinstance(emb_str, (list, np.ndarray)):
            # Already parsed
            return np.array(emb_str)
        return None
    except Exception as e:
        # Debug: uncomment to see parsing errors
        # print(f"Parse error: {str(e)[:50]} | Input: {str(emb_str)[:100]}")
        return None


def compute_intra_topic_variance(embeddings: List[np.ndarray], max_samples=1000) -> float:
    """
    Compute mean pairwise cosine distance within a topic.
    Lower = more compact = better topic coherence.
    
    Args:
        embeddings: List of embedding vectors
        max_samples: Maximum embeddings to sample (for speed)
    
    Returns:
        Mean cosine distance
    """
    if len(embeddings) < 2:
        return 0.0
    
    # Sample if too many (for speed)
    if len(embeddings) > max_samples:
        indices = np.random.choice(len(embeddings), max_samples, replace=False)
        embeddings = [embeddings[i] for i in indices]
    
    distances = []
    for i in range(len(embeddings)):
        for j in range(i + 1, min(i + 50, len(embeddings))):  # Limit pairs per item
            if embeddings[i] is not None and embeddings[j] is not None:
                dist = cosine(embeddings[i], embeddings[j])
                if not np.isnan(dist):
                    distances.append(dist)
    
    return np.mean(distances) if distances else 0.0


def load_topic_data(folder: Path, topic: str, embedding_col: str) -> Tuple[pd.DataFrame, Dict]:
    """
    Load topic CSV and compute statistics.
    
    Returns:
        df: DataFrame
        stats: Dict with statistics
    """
    filepath = folder / f"{topic}.csv"
    
    if not filepath.exists():
        print(f"   ⚠️  File not found: {filepath}")
        return None, None
    
    # Load data
    df = pd.read_csv(filepath)
    
    # Check required columns
    if topic not in df.columns:
        print(f"   ⚠️  Column '{topic}' not found in {filepath}")
        return None, None
    
    if embedding_col not in df.columns:
        print(f"   ⚠️  Column '{embedding_col}' not found in {filepath}")
        return None, None
    
    # Basic statistics
    stats = {
        'topic': topic,
        'num_rows': len(df),
        'mean_score': df[topic].mean(),
        'median_score': df[topic].median(),
        'std_score': df[topic].std(),
        'min_score': df[topic].min(),
        'max_score': df[topic].max()
    }
    
    return df, stats


print("✅ Helper functions loaded")
print("   - parse_embedding_string(): Parse embedding vectors")
print("   - compute_intra_topic_variance(): Calculate cosine distance compactness")
print("   - load_topic_data(): Load CSV and compute statistics")

✅ Helper functions loaded
   - parse_embedding_string(): Parse embedding vectors
   - compute_intra_topic_variance(): Calculate cosine distance compactness
   - load_topic_data(): Load CSV and compute statistics


In [ ]:
# ============================================================================
# STEP 1: LOAD DATA AND COMPUTE BASIC STATISTICS
# ============================================================================

print("=" * 100)
print("STEP 1: Loading Data and Computing Statistics")
print("=" * 100)

w3_stats = []
w5_stats = []
w3_dataframes = {}
w5_dataframes = {}

for topic in TOPICS:
    print(f"\n📊 Processing Topic: {topic}")
    print(f"{'-' * 100}")
    
    # Load W3 data
    print(f"   Loading W3 ({topic}.csv)...")
    w3_df, w3_stat = load_topic_data(W3_DIR, topic, 'w3_embedding')
    if w3_df is not None:
        w3_stats.append(w3_stat)
        w3_dataframes[topic] = w3_df
        print(f"      ✅ W3: {w3_stat['num_rows']:,} rows, mean={w3_stat['mean_score']:.4f}, std={w3_stat['std_score']:.4f}")
    
    # Load W5 data
    print(f"   Loading W5 ({topic}.csv)...")
    w5_df, w5_stat = load_topic_data(W5_DIR, topic, 'w5_embedding')
    if w5_df is not None:
        w5_stats.append(w5_stat)
        w5_dataframes[topic] = w5_df
        print(f"      ✅ W5: {w5_stat['num_rows']:,} rows, mean={w5_stat['mean_score']:.4f}, std={w5_stat['std_score']:.4f}")

print(f"\n{'=' * 100}")
print(f"✅ Loaded {len(w3_stats)} topics from W3, {len(w5_stats)} topics from W5")
print(f"{'=' * 100}\n")

STEP 1: Loading Data and Computing Statistics

📊 Processing Topic: War
----------------------------------------------------------------------------------------------------
   Loading W3 (War.csv)...
      ✅ W3: 446,373 rows, mean=0.3842, std=0.0625
   Loading W5 (War.csv)...
      ✅ W3: 446,373 rows, mean=0.3842, std=0.0625
   Loading W5 (War.csv)...
      ✅ W5: 453,438 rows, mean=0.3841, std=0.0624

📊 Processing Topic: Health
----------------------------------------------------------------------------------------------------
   Loading W3 (Health.csv)...
      ✅ W5: 453,438 rows, mean=0.3841, std=0.0624

📊 Processing Topic: Health
----------------------------------------------------------------------------------------------------
   Loading W3 (Health.csv)...


In [ ]:
# ============================================================================
# STEP 2: COMPUTE EMBEDDING COMPACTNESS (INTRA-TOPIC VARIANCE)
# ============================================================================

print("=" * 100)
print("STEP 2: Computing Embedding Compactness")
print("=" * 100)
print("Computing mean pairwise cosine distance within each topic...")
print("(Lower distance = more compact = better topic coherence)\n")

compactness_results = []

for topic in TOPICS:
    print(f"\n📐 Computing compactness: {topic}")
    print(f"{'-' * 100}")
    
    # W3 compactness
    w3_compactness = None
    if topic in w3_dataframes:
        df = w3_dataframes[topic]
        if 'w3_embedding' in df.columns:
            print(f"   Parsing W3 embeddings...")
            # Debug: show first embedding string
            first_emb = df['w3_embedding'].iloc[0]
            print(f"      Sample embedding (first 100 chars): {str(first_emb)[:100]}")
            
            embeddings = [parse_embedding_string(e) for e in df['w3_embedding'].head(1000)]
            embeddings = [e for e in embeddings if e is not None]
            
            if len(embeddings) > 1:
                w3_compactness = compute_intra_topic_variance(embeddings)
                print(f"      ✅ W3 compactness: {w3_compactness:.4f} (from {len(embeddings)} embeddings)")
            else:
                print(f"      ⚠️  Not enough valid embeddings (found {len(embeddings)} out of 1000 attempted)")
    
    # W5 compactness
    w5_compactness = None
    if topic in w5_dataframes:
        df = w5_dataframes[topic]
        if 'w5_embedding' in df.columns:
            print(f"   Parsing W5 embeddings...")
            # Debug: show first embedding string
            first_emb = df['w5_embedding'].iloc[0]
            print(f"      Sample embedding (first 100 chars): {str(first_emb)[:100]}")
            
            embeddings = [parse_embedding_string(e) for e in df['w5_embedding'].head(1000)]
            embeddings = [e for e in embeddings if e is not None]
            
            if len(embeddings) > 1:
                w5_compactness = compute_intra_topic_variance(embeddings)
                print(f"      ✅ W5 compactness: {w5_compactness:.4f} (from {len(embeddings)} embeddings)")
            else:
                print(f"      ⚠️  Not enough valid embeddings (found {len(embeddings)} out of 1000 attempted)")
        if 'w5_embedding' in df.columns:
            print(f"   Parsing W5 embeddings...")
            # Debug: show first embedding string
            first_emb = df['w5_embedding'].iloc[0]
            print(f"      Sample embedding (first 100 chars): {str(first_emb)[:100]}")
            
            embeddings = [parse_embedding_string(e) for e in df['w5_embedding'].head(1000)]
            embeddings = [e for e in embeddings if e is not None]
            
            if len(embeddings) > 1:
                w5_compactness = compute_intra_topic_variance(embeddings)
                print(f"      ✅ W5 compactness: {w5_compactness:.4f} (from {len(embeddings)} embeddings)")
            else:
                print(f"      ⚠️  Not enough valid embeddings (found {len(embeddings)} out of 1000 attempted)")
    
    compactness_results.append({
        'topic': topic,
        'w3_compactness': w3_compactness,
        'w5_compactness': w5_compactness,
        'better_compactness': 'W3' if (w3_compactness is not None and w5_compactness is not None and w3_compactness < w5_compactness) 
                                   else ('W5' if (w3_compactness is not None and w5_compactness is not None) else 'N/A')
    })

print(f"\n{'=' * 100}")
print(f"✅ Compactness computed for {len(compactness_results)} topics")
print(f"{'=' * 100}\n")

STEP 2: Computing Embedding Compactness
Computing mean pairwise cosine distance within each topic...
(Lower distance = more compact = better topic coherence)



NameError: name 'TOPICS' is not defined

In [ ]:
# ============================================================================
# STEP 3: OVERLAP ANALYSIS (SENTENCES IN MULTIPLE TOPICS)
# ============================================================================

print("=" * 100)
print("STEP 3: Overlap Analysis")
print("=" * 100)
print("Checking how many sentences appear in multiple topics within each model...\n")

def compute_overlap_within_folder(dataframes: Dict[str, pd.DataFrame], embedding_col: str) -> Dict:
    """
    Compute overlap statistics within a folder.
    
    Returns:
        Dict with overlap metrics
    """
    # Use main_sentence as unique identifier (or embedding string)
    sentence_topics = {}  # sentence -> list of topics
    
    for topic, df in dataframes.items():
        # Use main_sentence if available, otherwise embedding
        if 'main_sentence' in df.columns:
            sentences = df['main_sentence'].dropna().unique()
        elif embedding_col in df.columns:
            sentences = df[embedding_col].dropna().unique()
        else:
            continue
        
        for sent in sentences:
            if sent not in sentence_topics:
                sentence_topics[sent] = []
            sentence_topics[sent].append(topic)
    
    # Count overlaps
    total_sentences = len(sentence_topics)
    overlapping_sentences = sum(1 for topics in sentence_topics.values() if len(topics) > 1)
    overlap_percentage = (overlapping_sentences / total_sentences * 100) if total_sentences > 0 else 0
    
    # Distribution of overlap counts
    overlap_counts = {}
    for topics in sentence_topics.values():
        count = len(topics)
        overlap_counts[count] = overlap_counts.get(count, 0) + 1
    
    return {
        'total_sentences': total_sentences,
        'unique_sentences': total_sentences - overlapping_sentences,
        'overlapping_sentences': overlapping_sentences,
        'overlap_percentage': overlap_percentage,
        'overlap_distribution': overlap_counts
    }

# Compute W3 overlap
print("📊 Computing W3 overlap...")
w3_overlap = compute_overlap_within_folder(w3_dataframes, 'w3_embedding')
print(f"   Total sentences: {w3_overlap['total_sentences']:,}")
print(f"   Unique to one topic: {w3_overlap['unique_sentences']:,}")
print(f"   Appearing in multiple topics: {w3_overlap['overlapping_sentences']:,}")
print(f"   Overlap percentage: {w3_overlap['overlap_percentage']:.2f}%")
print(f"   Distribution: {w3_overlap['overlap_distribution']}")

print(f"\n{'-' * 100}\n")

# Compute W5 overlap
print("📊 Computing W5 overlap...")
w5_overlap = compute_overlap_within_folder(w5_dataframes, 'w5_embedding')
print(f"   Total sentences: {w5_overlap['total_sentences']:,}")
print(f"   Unique to one topic: {w5_overlap['unique_sentences']:,}")
print(f"   Appearing in multiple topics: {w5_overlap['overlapping_sentences']:,}")
print(f"   Overlap percentage: {w5_overlap['overlap_percentage']:.2f}%")
print(f"   Distribution: {w5_overlap['overlap_distribution']}")

print(f"\n{'=' * 100}")
print(f"📈 OVERLAP COMPARISON:")
print(f"   W3 Overlap: {w3_overlap['overlap_percentage']:.2f}%")
print(f"   W5 Overlap: {w5_overlap['overlap_percentage']:.2f}%")
print(f"   Better (lower overlap): {'W3' if w3_overlap['overlap_percentage'] < w5_overlap['overlap_percentage'] else 'W5'}")
print(f"{'=' * 100}\n")

In [ ]:
# ============================================================================
# STEP 4: CREATE COMPARISON SUMMARY TABLE
# ============================================================================

print("=" * 100)
print("STEP 4: Creating Comparison Summary Table")
print("=" * 100)

# Combine all statistics
comparison_data = []

for topic in TOPICS:
    # Find stats for this topic
    w3_stat = next((s for s in w3_stats if s['topic'] == topic), None)
    w5_stat = next((s for s in w5_stats if s['topic'] == topic), None)
    compact = next((c for c in compactness_results if c['topic'] == topic), None)
    
    if w3_stat and w5_stat:
        comparison_data.append({
            'Topic': topic,
            'W3_Rows': w3_stat['num_rows'],
            'W5_Rows': w5_stat['num_rows'],
            'W3_Mean': w3_stat['mean_score'],
            'W5_Mean': w5_stat['mean_score'],
            'W3_Median': w3_stat['median_score'],
            'W5_Median': w5_stat['median_score'],
            'W3_Std': w3_stat['std_score'],
            'W5_Std': w5_stat['std_score'],
            'W3_Min': w3_stat['min_score'],
            'W5_Min': w5_stat['min_score'],
            'W3_Max': w3_stat['max_score'],
            'W5_Max': w5_stat['max_score'],
            'W3_Compactness': compact['w3_compactness'] if compact else None,
            'W5_Compactness': compact['w5_compactness'] if compact else None
        })

comparison_df = pd.DataFrame(comparison_data)

# Add global overlap metrics
comparison_df['W3_Global_Overlap%'] = w3_overlap['overlap_percentage']
comparison_df['W5_Global_Overlap%'] = w5_overlap['overlap_percentage']

print("\n📋 COMPARISON SUMMARY TABLE:")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

In [ ]:
# ============================================================================
# STEP 5: DETERMINE BETTER MODEL PER TOPIC
# ============================================================================

print("\n" + "=" * 100)
print("STEP 5: Determining Better Model Per Topic")
print("=" * 100)

def determine_better_model(row):
    """
    Determine which model is better based on multiple criteria.
    
    Criteria:
    1. Higher mean similarity score (weight: 3)
    2. Lower standard deviation (weight: 2)
    3. Lower compactness/variance (weight: 2)
    4. (Global overlap is same for all topics, so not included here)
    
    Returns:
        'W3', 'W5', or 'TIE'
    """
    w3_score = 0
    w5_score = 0
    
    # 1. Higher mean similarity (weight: 3)
    if row['W3_Mean'] > row['W5_Mean']:
        w3_score += 3
    elif row['W5_Mean'] > row['W3_Mean']:
        w5_score += 3
    
    # 2. Lower std deviation (weight: 2) - more consistent
    if row['W3_Std'] < row['W5_Std']:
        w3_score += 2
    elif row['W5_Std'] < row['W3_Std']:
        w5_score += 2
    
    # 3. Lower compactness (weight: 2) - more coherent
    if pd.notna(row['W3_Compactness']) and pd.notna(row['W5_Compactness']):
        if row['W3_Compactness'] < row['W5_Compactness']:
            w3_score += 2
        elif row['W5_Compactness'] < row['W3_Compactness']:
            w5_score += 2
    
    # Determine winner
    if w3_score > w5_score:
        return 'W3'
    elif w5_score > w3_score:
        return 'W5'
    else:
        return 'TIE'

# Apply decision function
comparison_df['Better_Model'] = comparison_df.apply(determine_better_model, axis=1)

# Add decision breakdown
decision_breakdown = []
for _, row in comparison_df.iterrows():
    breakdown = []
    
    if row['W3_Mean'] > row['W5_Mean']:
        breakdown.append(f"Mean: W3 ({row['W3_Mean']:.4f} > {row['W5_Mean']:.4f})")
    else:
        breakdown.append(f"Mean: W5 ({row['W5_Mean']:.4f} > {row['W3_Mean']:.4f})")
    
    if row['W3_Std'] < row['W5_Std']:
        breakdown.append(f"Std: W3 ({row['W3_Std']:.4f} < {row['W5_Std']:.4f})")
    else:
        breakdown.append(f"Std: W5 ({row['W5_Std']:.4f} < {row['W3_Std']:.4f})")
    
    if pd.notna(row['W3_Compactness']) and pd.notna(row['W5_Compactness']):
        if row['W3_Compactness'] < row['W5_Compactness']:
            breakdown.append(f"Compact: W3 ({row['W3_Compactness']:.4f} < {row['W5_Compactness']:.4f})")
        else:
            breakdown.append(f"Compact: W5 ({row['W5_Compactness']:.4f} < {row['W3_Compactness']:.4f})")
    
    decision_breakdown.append('; '.join(breakdown))

comparison_df['Decision_Breakdown'] = decision_breakdown

# Display results
print("\n🏆 FINAL RECOMMENDATIONS:")
print("=" * 100)
for _, row in comparison_df.iterrows():
    print(f"\n📌 {row['Topic']}: {row['Better_Model']}")
    print(f"   {row['Decision_Breakdown']}")

print("\n" + "=" * 100)
print("📊 OVERALL SUMMARY:")
w3_wins = (comparison_df['Better_Model'] == 'W3').sum()
w5_wins = (comparison_df['Better_Model'] == 'W5').sum()
ties = (comparison_df['Better_Model'] == 'TIE').sum()

print(f"   W3 wins: {w3_wins} topics")
print(f"   W5 wins: {w5_wins} topics")
print(f"   Ties: {ties} topics")
print(f"\n   🎯 Overall recommendation: {'W3' if w3_wins > w5_wins else ('W5' if w5_wins > w3_wins else 'MIXED - Use topic-specific models')}")
print("=" * 100)

In [ ]:
# ============================================================================
# STEP 6: SAVE COMPARISON REPORT
# ============================================================================

print("\n" + "=" * 100)
print("STEP 6: Saving Comparison Report")
print("=" * 100)

# Save main comparison table
output_path = OUTPUT_DIR / "topic_model_comparison.csv"
comparison_df.to_csv(output_path, index=False)
print(f"\n✅ Main comparison saved to: {output_path}")

# Save detailed breakdown
detailed_output = OUTPUT_DIR / "topic_model_comparison_detailed.csv"
comparison_df.to_csv(detailed_output, index=False)
print(f"✅ Detailed comparison saved to: {detailed_output}")

# Save summary statistics
summary = {
    'analysis_date': pd.Timestamp.now().isoformat(),
    'w3_directory': str(W3_DIR),
    'w5_directory': str(W5_DIR),
    'topics_analyzed': TOPICS,
    'w3_wins': int(w3_wins),
    'w5_wins': int(w5_wins),
    'ties': int(ties),
    'overall_recommendation': 'W3' if w3_wins > w5_wins else ('W5' if w5_wins > w3_wins else 'MIXED'),
    'w3_overlap_percentage': float(w3_overlap['overlap_percentage']),
    'w5_overlap_percentage': float(w5_overlap['overlap_percentage']),
    'better_overlap': 'W3' if w3_overlap['overlap_percentage'] < w5_overlap['overlap_percentage'] else 'W5'
}

summary_path = OUTPUT_DIR / "topic_model_comparison_summary.json"
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Summary statistics saved to: {summary_path}")

# Create simplified table for quick reference
simple_comparison = comparison_df[['Topic', 'W3_Mean', 'W5_Mean', 'W3_Std', 'W5_Std', 
                                    'W3_Compactness', 'W5_Compactness', 'Better_Model']].copy()
simple_comparison.columns = ['Topic', 'W3_Mean', 'W5_Mean', 'W3_Std', 'W5_Std', 
                              'W3_Compact', 'W5_Compact', 'Recommended']

simple_path = OUTPUT_DIR / "topic_model_comparison_simple.csv"
simple_comparison.to_csv(simple_path, index=False)
print(f"✅ Simplified comparison saved to: {simple_path}")

print("\n" + "=" * 100)
print("🎉 COMPARISON ANALYSIS COMPLETE!")
print("=" * 100)
print(f"\n📁 Output files:")
print(f"   1. {output_path.name} - Full comparison table")
print(f"   2. {detailed_output.name} - Detailed breakdown")
print(f"   3. {summary_path.name} - JSON summary")
print(f"   4. {simple_path.name} - Quick reference table")
print("\n" + "=" * 100)

In [ ]:
# ============================================================================
# STEP 7: VISUALIZATION - COMPARISON CHARTS
# ============================================================================

print("\n" + "=" * 100)
print("STEP 7: Creating Visualization Charts")
print("=" * 100)

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Topic Model Comparison: W3 vs W5 Embeddings', fontsize=18, fontweight='bold', y=0.995)

# 1. Mean Similarity Scores
ax1 = axes[0, 0]
x = np.arange(len(TOPICS))
width = 0.35
ax1.bar(x - width/2, comparison_df['W3_Mean'], width, label='W3', color='#3498db', alpha=0.8, edgecolor='black')
ax1.bar(x + width/2, comparison_df['W5_Mean'], width, label='W5', color='#e74c3c', alpha=0.8, edgecolor='black')
ax1.set_xlabel('Topic', fontweight='bold', fontsize=11)
ax1.set_ylabel('Mean Similarity Score', fontweight='bold', fontsize=11)
ax1.set_title('Mean Similarity Scores by Topic', fontweight='bold', fontsize=13)
ax1.set_xticks(x)
ax1.set_xticklabels(TOPICS, rotation=45, ha='right')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# 2. Standard Deviation
ax2 = axes[0, 1]
ax2.bar(x - width/2, comparison_df['W3_Std'], width, label='W3', color='#3498db', alpha=0.8, edgecolor='black')
ax2.bar(x + width/2, comparison_df['W5_Std'], width, label='W5', color='#e74c3c', alpha=0.8, edgecolor='black')
ax2.set_xlabel('Topic', fontweight='bold', fontsize=11)
ax2.set_ylabel('Standard Deviation', fontweight='bold', fontsize=11)
ax2.set_title('Standard Deviation by Topic (Lower = More Consistent)', fontweight='bold', fontsize=13)
ax2.set_xticks(x)
ax2.set_xticklabels(TOPICS, rotation=45, ha='right')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# 3. Compactness
ax3 = axes[1, 0]
w3_compact = comparison_df['W3_Compactness'].fillna(0)
w5_compact = comparison_df['W5_Compactness'].fillna(0)
ax3.bar(x - width/2, w3_compact, width, label='W3', color='#3498db', alpha=0.8, edgecolor='black')
ax3.bar(x + width/2, w5_compact, width, label='W5', color='#e74c3c', alpha=0.8, edgecolor='black')
ax3.set_xlabel('Topic', fontweight='bold', fontsize=11)
ax3.set_ylabel('Compactness Score (Cosine Distance)', fontweight='bold', fontsize=11)
ax3.set_title('Embedding Compactness by Topic (Lower = More Coherent)', fontweight='bold', fontsize=13)
ax3.set_xticks(x)
ax3.set_xticklabels(TOPICS, rotation=45, ha='right')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# 4. Winner Summary
ax4 = axes[1, 1]
winner_counts = comparison_df['Better_Model'].value_counts()
colors_map = {'W3': '#3498db', 'W5': '#e74c3c', 'TIE': '#95a5a6'}
bars = ax4.bar(winner_counts.index, winner_counts.values, 
               color=[colors_map.get(x, '#95a5a6') for x in winner_counts.index],
               alpha=0.8, edgecolor='black')
ax4.set_xlabel('Model', fontweight='bold', fontsize=11)
ax4.set_ylabel('Number of Topics', fontweight='bold', fontsize=11)
ax4.set_title('Overall Winner Count', fontweight='bold', fontsize=13)
ax4.set_ylim(0, len(TOPICS))
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontweight='bold', fontsize=12)

plt.tight_layout()

# Save figure
viz_path = OUTPUT_DIR / "topic_model_comparison_visualization.png"
plt.savefig(viz_path, dpi=300, bbox_inches='tight')
print(f"\n✅ Visualization saved to: {viz_path}")

plt.show()

print("\n" + "=" * 100)
print("✅ VISUALIZATION COMPLETE!")
print("=" * 100)